# Revision-4 experiments

Additional experiments requested in the fourth review round of
*Tabular Foundation Models and Few-Shot LLMs for IoT Intrusion Detection*.
The notebook reuses the dataset loaders, protocol helpers and Axis-2 code path of the
released campaigns, so every new number is produced under the protocol of the manuscript.

| # | Experiment | Reviewer item | Needs | Approx. cost |
|---|---|---|---|---|
| 5 | $2\times2$ size-by-balance factorial on **CIC-UNSW-NB15 V1** (XGBoost, 10 seeds) | R1 factorial replication | CPU, repository | 30–60 min |
| 6 | Data-rich tree references on the **Axis-2 evaluation rows** (strong RF and XGBoost; 10 seeds) | R1 data-rich baseline | CPU, Kaggle table | 10–20 min |
| 4b | ManyClass sensitivity extended from 3 to **10 seeds** | R1 ManyClass seeds | GPU, `HF_TOKEN`, repository | ~28 TabPFN runs (≈ 30 min each on the round-3 hardware) |
| 7 | Six-model Axis-2 panel on a **second variant: CIC-UNSW-NB15 V1** | R1 Axis-2 generality | `NEBIUS_API_KEY`, repository | ≈ USD 270–320 at the round-3 prices (switches below reduce it) |
| A | Analysis: LaTeX tables + prose numbers for the manuscript | — | CPU only | seconds |

Each experiment has a switch and is resumable: results are appended after every
unit of work and completed units are skipped on restart. **Run the Analysis section
at the end whichever experiments were executed**; it also copies the generated tables
into the manuscript `generated/` folder when `MANUSCRIPT_DIR` is set.

Colab secrets used (key icon, left sidebar): `NEBIUS_API_KEY`, `HF_TOKEN`, `KAGGLE_USERNAME`, `KAGGLE_KEY`.

In [ ]:
# ============================ SWITCHES ============================
RUN_2X2_UNSW       = True    # experiment 5 — CPU only, needs repository loaders
RUN_AXIS2_DATARICH = True    # experiment 6 — CPU only, needs Kaggle table (no API key)
RUN_MANYCLASS_10   = True   # experiment 4b — GPU + HF_TOKEN + repository (long)
RUN_AXIS2_SECOND   = True   # experiment 7 — NEBIUS_API_KEY + repository (paid)

SEEDS        = list(range(10))
OUT_DIR      = "revision4_outputs"
REPO_DIR     = "ids_benchmark_release"      # clone of the released repository (contains src/ and results/)
REPO_URL     = ""   # set to the anonymous review repository URL, or pre-clone into REPO_DIR
MANUSCRIPT_DIR = None                       # e.g. "/content/drive/MyDrive/llm_ids/manuscript" to copy tables

# Experiment 7 scope (cost control). Full protocol = all six models, ten seeds.
SECOND_VARIANT_MODELS = ["GLM-5.2", "DeepSeek-V4-Pro", "Kimi-K3", "MiniMax-M3",
                         "Qwen3.5-397B-A17B", "Nemotron-3-Ultra-550b-a55b"]
SECOND_VARIANT_SEEDS  = SEEDS

NEEDS_REPO   = RUN_2X2_UNSW or RUN_MANYCLASS_10 or RUN_AXIS2_SECOND
NEEDS_API    = RUN_AXIS2_SECOND
NEEDS_MODELS = RUN_MANYCLASS_10
NEEDS_KAGGLE = RUN_AXIS2_DATARICH
print(f"repo={NEEDS_REPO} api={NEEDS_API} model stack={NEEDS_MODELS} kaggle={NEEDS_KAGGLE}")

## 0. Dependencies and helpers

Identical to the released sensitivity notebook (pinned model stack for TabPFN), plus a Student-$t$ interval and a Holm helper for the new analyses.

In [ ]:
import subprocess, sys, importlib

def pip(pkgs):
    missing = []
    alias = {"scikit-learn":"sklearn","huggingface-hub":"huggingface_hub",
             "pytorch-tabnet":"pytorch_tabnet"}
    for spec in pkgs:
        mod = alias.get(spec.split("==")[0], spec.split("==")[0].replace("-","_"))
        try: importlib.import_module(mod)
        except Exception: missing.append(spec)
    if not missing:
        print(f"[install] present: {', '.join(pkgs)}"); return
    r = subprocess.run([sys.executable,"-m","pip","install","-q"]+missing,
                       capture_output=True, text=True)
    if r.returncode and "externally-managed" in (r.stderr or ""):
        r = subprocess.run([sys.executable,"-m","pip","install","-q",
                            "--break-system-packages"]+missing, capture_output=True, text=True)
    print(f"[install] {'ok' if r.returncode==0 else 'FAILED'}: {', '.join(missing)}")
    if r.returncode:
        print("   ", (r.stderr or "").strip().splitlines()[-1][:160])

pip(["pandas","numpy","scikit-learn","xgboost","pyarrow"])
if NEEDS_API:    pip(["openai"])
if NEEDS_KAGGLE or NEEDS_API: pip(["kagglehub","pyarrow"])
if NEEDS_REPO:   pip(["kagglehub","huggingface-hub"])
if NEEDS_MODELS: pip(["transformers==4.57.1","tokenizers==0.22.1","safetensors==0.7.0",
                      "huggingface-hub==0.36.0","accelerate==1.11.0","peft==0.18.0",
                      "tabpfn==6.0.6","tabpfn-extensions==0.2.1","tabpfn-common-utils==0.2.10",
                      "tabicl==0.1.4","pytorch-tabnet==4.1.0"])

In [ ]:
import gc, importlib, itertools, json, os, re, shutil, sys, time, types
from pathlib import Path
import numpy as np, pandas as pd

OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)

# --- secrets, from Colab or the environment ------------------------------
try:
    from google.colab import userdata
    for k in ("NEBIUS_API_KEY","HF_TOKEN","KAGGLE_USERNAME","KAGGLE_KEY"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception: pass
except Exception: pass
if NEEDS_API:
    assert os.environ.get("NEBIUS_API_KEY"), "NEBIUS_API_KEY is required for experiment 7"

def sf(d):
    """Exact two-sided paired sign-flip p. Floor at n pairs is 2**-(n-1)."""
    d = np.asarray(d, float); o = abs(d.mean())
    return float(np.mean([abs(np.mean(d*np.array(x))) >= o-1e-15
                          for x in itertools.product((-1,1), repeat=len(d))]))
def ci(d, n=10000):
    d = np.asarray(d, float); b = np.random.default_rng(0).choice(d,(n,len(d))).mean(1)
    return tuple(np.percentile(b,[2.5,97.5]))
def save(df, name):
    df.to_csv(OUT/name, index=False); print(f"[saved] {name} ({len(df)} rows)"); return df
print("output:", OUT.resolve())

from scipy import stats as _st
def tci(x, conf=0.95):
    """Student-t interval for a mean (used when fewer than ten seeds are available)."""
    x = np.asarray(x, float); n = len(x)
    if n < 2: return (np.nan, np.nan)
    h = _st.t.ppf(0.5+conf/2, n-1) * x.std(ddof=1)/np.sqrt(n)
    return (float(x.mean()-h), float(x.mean()+h))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); out = np.empty_like(p); run = 0.0
    for i, idx in enumerate(o):
        run = max(run, (len(p)-i)*p[idx]); out[idx] = min(1.0, run)
    return out
print("tci, holm ready")

## 1. Repository and dataset loaders *(experiments 5, 4b and 7)*

The released repository provides the persisted results the notebook verifies and resumes from (`results/axis2`, `results/sensitivity`, `results/derived`). The public release does not ship the dataset-loader package of the development tree, so the loaders are **vendored** in the next cell as a faithful transcription of `src/data`: same cleaning steps, same processed-file names, same cache paths. Each cleaned table is checked against the row and class counts stated in the manuscript (CIC-IDS2017: 2,522,362 rows, 15 native / 9 grouped classes; CIC-UNSW-NB15: 3,540,241 rows, 10 classes); a mismatch aborts (`STRICT_COUNTS`). The two shims for the pinned model stack are unchanged from the released sensitivity notebook.

In [ ]:
if NEEDS_REPO:
    # ---- released repository: persisted results and analysis scripts ------------
    # The anonymous review repository ships the
    # audited evidential record (results/), the docs and the verification
    # scripts; the dataset-loader package of the development tree is NOT part of
    # the release. The loaders are therefore vendored below, as a faithful
    # transcription of src/data (same cleaning steps, same processed-file names,
    # same cache locations), so this notebook is self-contained.
    if not Path(REPO_DIR, "results").exists():
        subprocess.run(f"git clone -q {REPO_URL} {REPO_DIR} || true",
                       shell=True, check=False)
    REPO = Path(REPO_DIR).resolve()
    assert (REPO/"results").exists(), (
        f"repository not found at {REPO}; set REPO_URL "
        f"to the anonymous review repository URL or pre-clone into REPO_DIR")
    print(f"repository: {REPO}")

    # shim 1: tabpfn 8.x renamed the preprocessor factory
    try:
        import tabpfn.inference_config as _ic
        for _old, _new in [("default_classifier_preprocessor_configs",
                            ["v2_5_classifier_preprocessor_configs","v2_classifier_preprocessor_configs"]),
                           ("default_regressor_preprocessor_configs",
                            ["v2_5_regressor_preprocessor_configs","v2_regressor_preprocessor_configs"])]:
            if not hasattr(_ic,_old):
                for _n in _new:
                    if hasattr(_ic,_n): setattr(_ic,_old,getattr(_ic,_n)); break
    except ImportError: pass

    # shim 2: huggingface_hub >= 1.0 removed HfFolder
    try:
        import huggingface_hub as _hh
        if not hasattr(_hh,"HfFolder"):
            class HfFolder:
                path_token = str(Path.home()/".cache/huggingface/token")
                @classmethod
                def get_token(cls):
                    try: return _hh.get_token()
                    except Exception: return None
                @classmethod
                def save_token(cls,t):
                    p=Path(cls.path_token); p.parent.mkdir(parents=True,exist_ok=True); p.write_text(t or "")
                @classmethod
                def delete_token(cls): Path(cls.path_token).unlink(missing_ok=True)
            _hh.HfFolder = HfFolder
    except ImportError: pass

    # ---- vendored dataset loaders (transcription of src/data) --------------------
    import requests as _rq

    DATA_DIR = "data"
    STRICT_COUNTS = True     # verify cleaned-table sizes against the manuscript
    UNSW_LOCAL_CSV = None    # set to a local CICFlowMeter_out.csv to skip the download
    CICUNSW_URL = "http://cicresearch.ca/CICDataset/CIC-UNSW/Dataset/CICFlowMeter_out.csv"

    CIC_IDS2017_CLASSES_MAPPING = {
        'BENIGN': 'BENIGN', 'DDoS': 'DDoS', 'DoS Hulk': 'DoS',
        'DoS GoldenEye': 'DoS', 'DoS slowloris': 'DoS', 'DoS Slowhttptest': 'DoS',
        'PortScan': 'Port Scan', 'FTP-Patator': 'Brute Force',
        'SSH-Patator': 'Brute Force', 'Bot': 'Bot',
        'Web Attack  Brute Force': 'Web Attack',
        'Web Attack  Sql Injection': 'Web Attack',
        'Web Attack  XSS': 'Web Attack',
        'Infiltration': 'Infiltration', 'Heartbleed': 'Heartbleed'}

    # expected cleaned-table sizes (Section 3.2 of the manuscript)
    _EXPECTED = {("cic", False): (2_522_362, 15), ("cic", True): (2_522_362, 9),
                 ("unsw", None): (3_540_241, 10)}

    def _features_correction(d):
        d.rename(columns={c: str(c).strip().replace(" ", "_") for c in d.columns},
                 inplace=True)
    def _ascii_correction(d, col):
        d[col] = d[col].apply(lambda x: re.sub(r'[^\x00-\x7F]+', '', str(x)))
    def _downcast(d):
        for col in d.columns:
            t = d[col].dtype
            if t != object:
                lo, hi = d[col].min(), d[col].max()
                if 'float' in str(t) and lo > np.finfo(np.float32).min and hi < np.finfo(np.float32).max:
                    try: d[col] = d[col].astype(np.float32)
                    except ValueError: pass
                elif 'int' in str(t) and lo > np.iinfo(np.int32).min and hi < np.iinfo(np.int32).max:
                    d[col] = d[col].astype(np.int32)
    def _drop_single_unique(d):
        nu = d.nunique()
        return d[nu[nu > 1].index]
    def _check_counts(tag, d):
        exp = _EXPECTED.get(tag)
        if exp is None: return
        rows, classes = exp
        ok = (len(d) == rows) and (d["Label"].nunique() == classes)
        msg = (f"{tag}: cleaned table has {len(d):,} rows / {d['Label'].nunique()} classes, "
               f"manuscript states {rows:,} / {classes}")
        if ok: print("  count check OK -", msg)
        elif STRICT_COUNTS:
            raise AssertionError(msg + "  (raw sources changed? set STRICT_COUNTS=False only "
                                       "if you understand why, since every number depends on this table)")
        else: print("  WARNING -", msg)

    class CICIDS2017:
        """Vendored transcription of src.data CICIDS2017 (pca / classes_mapping config)."""
        dataset_folder = "MachineLearningCVE"
        def __init__(self, pca=True, classes_mapping=True, **kw):
            self.default_config = {"classes_mapping": classes_mapping, "pca": pca}
            self.raw_data_dir = os.path.abspath(os.path.join(DATA_DIR, "CIC-IDS2017", self.dataset_folder))
            self.processed_data_dir = os.path.abspath(os.path.join(DATA_DIR, "CIC-IDS2017", "processed"))
            name = "cic_ids2017_" + "_".join(f"{k}({v})" for k, v in sorted(self.default_config.items()))
            self.file_path = os.path.join(self.processed_data_dir, f"{name}.csv")
            self.data = None
        def _collect(self):
            if os.path.isfile(self.file_path):
                self.data = pd.read_csv(self.file_path); return
            raw = Path(self.raw_data_dir)
            files = [raw/f for f in CIC_FILES if (raw/f).is_file()]
            files += sorted(p for p in raw.glob("*.csv") if p not in files)
            assert len(files) >= 8, (f"expected the 8 raw CIC-IDS2017 CSVs in {raw}; "
                                     f"found {len(files)} (run _prefetch_cic / set LOCAL_CIC)")
            # deterministic capture-day order (the order _prefetch_cic writes them)
            self.data = pd.concat((pd.read_csv(f, encoding="latin-1") for f in files),
                                  ignore_index=False)
        def _preprocess(self):
            _features_correction(self.data)
            _ascii_correction(self.data, "Label")
            self.data.drop_duplicates(inplace=True)
            self.data.replace([np.inf, -np.inf], np.nan, inplace=True)
            for c in ("Flow_Bytes/s", "Flow_Packets/s"):
                if c in self.data.columns:
                    self.data[c] = self.data[c].fillna(self.data[c].median())
            self.data.dropna(inplace=True)
            _downcast(self.data)
            self.data = _drop_single_unique(self.data)
            if self.default_config["pca"]:
                raise NotImplementedError("the vendored loader is used with pca=False only; "
                                          "PCA variants are built by transform(use_pca=True)")
            if self.default_config["classes_mapping"]:
                self.data["Label"] = self.data["Label"].map(CIC_IDS2017_CLASSES_MAPPING)
        def load(self):
            fresh = not os.path.isfile(self.file_path)
            self._collect()
            if fresh:
                self._preprocess()
                Path(self.processed_data_dir).mkdir(parents=True, exist_ok=True)
                self.data.to_csv(self.file_path, index=False)
            _check_counts(("cic", self.default_config["classes_mapping"]), self.data)

    class CIC_UNSW:
        """Vendored transcription of src.data CIC_UNSW (CICFlowMeter re-derivation)."""
        dataset_folder = "CIC-UNSW"
        def __init__(self, pca=True, **kw):
            self.config = {"pca": pca}
            self.raw_data_dir = os.path.abspath(os.path.join(DATA_DIR, self.dataset_folder, "raw"))
            self.processed_data_dir = os.path.abspath(os.path.join(DATA_DIR, self.dataset_folder, "processed"))
            name = "cic_unsw_" + "_".join(f"{k}({v})" for k, v in sorted(self.config.items()))
            self.file_path = os.path.join(self.processed_data_dir, f"{name}.csv")
            self.data = None
        def _download(self):
            os.makedirs(self.raw_data_dir, exist_ok=True)
            csv_path = os.path.join(self.raw_data_dir, os.path.basename(CICUNSW_URL))
            if os.path.isfile(csv_path): return
            if UNSW_LOCAL_CSV:
                shutil.copy2(UNSW_LOCAL_CSV, csv_path); return
            print(f"  downloading {CICUNSW_URL} (~1.9 GB, one time)...")
            r = _rq.get(CICUNSW_URL, stream=True, timeout=120)
            assert r.status_code == 200, f"download failed: HTTP {r.status_code}"
            with open(csv_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
            head = open(csv_path, "rb").read(200)
            assert os.path.getsize(csv_path) > 10_000_000 and not head.lstrip().startswith(b"<"), (
                "the CIC host returned an error page instead of the CSV; retry, or place the "
                "file manually and set UNSW_LOCAL_CSV")
        def _clean(self):
            files = [f for f in os.listdir(self.raw_data_dir) if f.endswith(".csv")]
            assert len(files) == 1, f"expected one raw CSV in {self.raw_data_dir}, got {len(files)}"
            df = pd.read_csv(os.path.join(self.raw_data_dir, files[0]))
            labels = df["Label"]
            df = df.select_dtypes(include=["number"])
            df["Label"] = labels
            return df
        def _preprocess(self):
            _features_correction(self.data)
            _downcast(self.data)
            self.data = _drop_single_unique(self.data)
            if self.config["pca"]:
                raise NotImplementedError("the vendored loader is used with pca=False only")
        def load(self):
            if os.path.isfile(self.file_path):
                self.data = pd.read_csv(self.file_path)
            else:
                self._download()
                self.data = self._clean()
                self._preprocess()
                Path(self.processed_data_dir).mkdir(parents=True, exist_ok=True)
                self.data.to_csv(self.file_path, index=False)
            _check_counts(("unsw", None), self.data)

    print("vendored loaders ready (CICIDS2017, CIC_UNSW)")
    print(f"pandas {pd.__version__}" + ("  (pin pandas<3 for numbers that go in the paper)"
                                        if int(pd.__version__.split('.')[0])>=3 else ""))
else:
    print("repository not needed for the enabled experiments")


In [ ]:
# ============================================================
# PATCH — robust CIC-UNSW-NB15 raw-file acquisition
# Run after the vendored-loader cell and before Experiment 5.
# ============================================================
#
# The original CIC URL sometimes returns an HTML error page with HTTP 200.
# The original loader then leaves that response at the .csv path and treats it
# as cached on the next run. This replacement:
#   1. rejects/removes that stale response;
#   2. downloads the same CICFlowMeter_out.csv from a pinned HF dataset commit;
#   3. verifies the exact byte count and SHA-256; and
#   4. makes _clean read that exact file, rather than any arbitrary CSV nearby.

if NEEDS_REPO:
    from pathlib import Path
    import hashlib
    import shutil

    UNSW_HF_REPO = "bencorn/CIC-UNSW-NB15"
    UNSW_HF_REVISION = "8b5e371bf1cfb32683a958f62acacc6be19b5c58"
    UNSW_HF_FILE = "CICFlowMeter_out.csv"
    UNSW_EXPECTED_BYTES = 1_914_305_081
    UNSW_EXPECTED_SHA256 = (
        "025a94239f29521e154be3a65d65268404d4c58600b66b94a4235eae87dbd8b5"
    )

    def _sha256_stream(path, chunk_size=8 << 20):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for chunk in iter(lambda: handle.read(chunk_size), b""):
                digest.update(chunk)
        return digest.hexdigest()

    def _looks_like_unsw_csv(path, *, require_exact_size=True):
        path = Path(path)
        if not path.is_file():
            return False, "missing"
        size = path.stat().st_size
        if require_exact_size and size != UNSW_EXPECTED_BYTES:
            return False, f"wrong size ({size:,} bytes)"
        if size < 10_000_000:
            return False, f"too small ({size:,} bytes)"
        with path.open("rb") as handle:
            head = handle.read(65_536).lstrip()
        if head.startswith((b"<", b"<!DOCTYPE", b"<?xml")):
            return False, "HTML/XML response"
        if b"Label" not in head:
            return False, "CSV header does not contain Label"
        return True, "ok"

    def _patched_unsw_download(self):
        from huggingface_hub import hf_hub_download

        raw_dir = Path(self.raw_data_dir)
        raw_dir.mkdir(parents=True, exist_ok=True)
        target = raw_dir / UNSW_HF_FILE
        marker = raw_dir / f"{UNSW_HF_FILE}.sha256-ok"

        # Honour a manually supplied file, but validate it before copying.
        if UNSW_LOCAL_CSV:
            source = Path(UNSW_LOCAL_CSV).expanduser().resolve()
            ok, why = _looks_like_unsw_csv(source)
            if not ok:
                raise AssertionError(
                    f"UNSW_LOCAL_CSV is not the expected CICFlowMeter_out.csv: {why}; "
                    f"path={source}"
                )
            if source != target.resolve():
                tmp = target.with_suffix(target.suffix + ".copying")
                tmp.unlink(missing_ok=True)
                shutil.copy2(source, tmp)
                tmp.replace(target)
            marker.unlink(missing_ok=True)

        ok, why = _looks_like_unsw_csv(target)
        if not ok:
            # A marker belongs only to the exact file that was previously hashed.
            marker.unlink(missing_ok=True)
            if target.exists():
                bad = target.with_name(target.name + ".bad-response")
                bad.unlink(missing_ok=True)
                target.replace(bad)
                print(f"  removed invalid cached response ({why}): {bad}")

            print(
                "  downloading pinned mirror "
                f"{UNSW_HF_REPO}/{UNSW_HF_FILE} (~1.9 GB, one time)..."
            )

            last_error = None
            for force in (False, True):
                try:
                    downloaded = Path(
                        hf_hub_download(
                            repo_id=UNSW_HF_REPO,
                            filename=UNSW_HF_FILE,
                            repo_type="dataset",
                            revision=UNSW_HF_REVISION,
                            local_dir=str(raw_dir),
                            force_download=force,
                        )
                    )
                    if downloaded.resolve() != target.resolve():
                        tmp = target.with_suffix(target.suffix + ".copying")
                        tmp.unlink(missing_ok=True)
                        shutil.copy2(downloaded, tmp)
                        tmp.replace(target)

                    ok, why = _looks_like_unsw_csv(target)
                    if ok:
                        break
                    last_error = AssertionError(why)
                    target.unlink(missing_ok=True)
                except Exception as exc:
                    last_error = exc
                    target.unlink(missing_ok=True)
            else:
                raise RuntimeError(
                    "Could not obtain a valid CICFlowMeter_out.csv from the pinned mirror"
                ) from last_error

        # Hash once per valid cached copy. The marker avoids hashing 1.9 GB on
        # every rerun; deleting/replacing the CSV invalidates the marker above.
        marker_ok = (
            marker.is_file()
            and marker.read_text(encoding="utf-8").strip() == UNSW_EXPECTED_SHA256
        )
        if not marker_ok:
            print("  verifying CIC-UNSW-NB15 SHA-256...")
            observed = _sha256_stream(target)
            if observed != UNSW_EXPECTED_SHA256:
                target.unlink(missing_ok=True)
                marker.unlink(missing_ok=True)
                raise AssertionError(
                    "CICFlowMeter_out.csv SHA-256 mismatch: "
                    f"observed {observed}, expected {UNSW_EXPECTED_SHA256}"
                )
            marker.write_text(UNSW_EXPECTED_SHA256 + "\n", encoding="utf-8")

        print(
            f"  raw CIC-UNSW file ready: {target} "
            f"({target.stat().st_size:,} bytes; SHA-256 verified)"
        )

    def _patched_unsw_clean(self):
        csv_path = Path(self.raw_data_dir) / UNSW_HF_FILE
        ok, why = _looks_like_unsw_csv(csv_path)
        if not ok:
            raise AssertionError(
                f"expected a validated {UNSW_HF_FILE} at {csv_path}, but found: {why}"
            )
        df = pd.read_csv(csv_path)
        if "Label" not in df.columns:
            raise KeyError(
                f"Label column missing from {csv_path}; first columns: {list(df.columns[:8])}"
            )
        labels = df["Label"].copy()
        df = df.select_dtypes(include=["number"]).copy()
        df["Label"] = labels.to_numpy()
        return df

    CIC_UNSW._download = _patched_unsw_download
    CIC_UNSW._clean = _patched_unsw_clean

    # Clear only an in-memory failed/partial UNSW entry, if one somehow exists.
    if "_cache" in globals():
        _cache.pop(("unsw", False), None)
        _cache.pop(("unsw", None), None)

    print("CIC-UNSW download patch installed")
else:
    print("CIC-UNSW patch not needed because repository-dependent experiments are disabled")


In [ ]:
CIC_FILES = ["Monday-WorkingHours.pcap_ISCX.csv","Tuesday-WorkingHours.pcap_ISCX.csv",
 "Wednesday-workingHours.pcap_ISCX.csv","Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
 "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv","Friday-WorkingHours-Morning.pcap_ISCX.csv",
 "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv","Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"]
LOCAL_CIC = None      # set to a folder with the 8 raw CSVs to skip the network

def _prefetch_cic(ds):
    """Place the raw CSVs where the loader expects them.

    The repository's own _download validates only the HTTP status code, so an
    error page from the CIC host is written as a .zip and zipfile then fails.
    Creating processed_data_dir also stops _collect from attempting a download.
    """
    raw, proc = Path(ds.raw_data_dir), Path(ds.processed_data_dir)
    raw.mkdir(parents=True, exist_ok=True); proc.mkdir(parents=True, exist_ok=True)
    if Path(ds.file_path).is_file() or len(list(raw.glob("*.csv"))) >= 8: return
    if LOCAL_CIC:
        for p in Path(LOCAL_CIC).rglob("*.csv"): shutil.copy2(p, raw/p.name)
        if len(list(raw.glob("*.csv"))) >= 8: return
    from huggingface_hub import hf_hub_download
    for fn in CIC_FILES:
        shutil.copy2(hf_hub_download(repo_id="c01dsnap/CIC-IDS2017", filename=fn,
                                     repo_type="dataset"), raw/fn)
    print(f"  fetched {len(list(raw.glob('*.csv')))} raw CSVs")

_cache = {}
def load_base(which, grouped):
    """Cleaned table through the repository loader. PCA is never applied here."""
    key = (which, grouped)
    if key in _cache: return _cache[key]
    ds = CICIDS2017(pca=False, classes_mapping=grouped) if which=="cic" else CIC_UNSW(pca=False)
    if which == "cic": _prefetch_cic(ds)
    ds.load()
    df = ds.data.copy().reset_index(drop=True)
    for c in df.columns:
        if df[c].dtype == np.float64: df[c] = df[c].astype(np.float32)
    _cache[key] = df; gc.collect()
    print(f"  {which} grouped={grouped}: {len(df):,} rows, {df.Label.nunique()} classes")
    return df

# ---- protocol, identical to the reported runs ---------------------------
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

TEST_SIZE, FIXED_SEED = 0.30, 2026
KEEP_FRACTION, CAP_MIN = 0.05, 1500
RARE_SHARE = 0.005

def set_seed(s):
    os.environ["PYTHONHASHSEED"]=str(s); np.random.seed(s)
    try:
        import torch, random; random.seed(s); torch.manual_seed(s)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    except Exception: pass

def split(df, seed):
    """One stratified split, fixed across seeds, as in the reported protocol."""
    vc = df.Label.value_counts(); tiny = list(vc[vc<2].index)
    if tiny: df = df[~df.Label.isin(tiny)]
    idx = np.arange(len(df))
    tr, te = train_test_split(idx, test_size=TEST_SIZE, random_state=FIXED_SEED,
                              stratify=df.Label.to_numpy())
    return df.iloc[tr], df.iloc[te], {"split_seed": FIXED_SEED}

def balanced(df, n, seed):
    r = np.random.default_rng(seed)
    return pd.concat([p.iloc[r.choice(len(p), min(len(p), n), replace=False)]
                      for _, p in df.groupby("Label", sort=True)]).sample(frac=1, random_state=seed)

def cap_test(test, seed):
    """Class-stratified subsample with inverse sampling weights."""
    r = np.random.default_rng(seed); keep=[]; full = test.Label.value_counts()
    for c, p in test.groupby("Label", sort=True):
        k = min(len(p), max(CAP_MIN, int(np.ceil(len(p)*KEEP_FRACTION))))
        keep.append(p.iloc[r.choice(len(p), k, replace=False)] if k < len(p) else p)
    sub = pd.concat(keep); kept = sub.Label.value_counts()
    return sub, {c: float(full[c])/float(kept[c]) for c in kept.index}

def transform(train, test, use_pca=False):
    xtr = train.drop(columns=["Label"]).select_dtypes(include=[np.number]).astype(np.float32)
    xte = test.reindex(columns=xtr.columns).astype(np.float32)
    imp = SimpleImputer(strategy="median"); a=imp.fit_transform(xtr); b=imp.transform(xte)
    vt = VarianceThreshold(0.0); a=vt.fit_transform(a); b=vt.transform(b)
    names = [c for c,k in zip(xtr.columns, vt.get_support()) if k]
    if use_pca:
        sc=StandardScaler(); a=sc.fit_transform(a); b=sc.transform(b)
        k=max(1,min(len(names)//2,len(a)-1))
        pca=IncrementalPCA(n_components=k,batch_size=max(1000,2*k))
        a=pca.fit_transform(a); b=pca.transform(b); names=[f"PC{i+1}" for i in range(k)]
    F=lambda m: pd.DataFrame(np.asarray(m,np.float32),columns=names).reset_index(drop=True)
    return F(a), F(b)

def weighted_metrics(y_true, y_pred, weights=None):
    """Per-class metrics from a confusion matrix rescaled by class weights."""
    labs = sorted(set(map(str,y_true)) | set(map(str,y_pred)))
    idx = {c:i for i,c in enumerate(labs)}
    cm = np.zeros((len(labs),len(labs))); w = weights or {}
    for a,b in zip(map(str,y_true), map(str,y_pred)):
        cm[idx[a], idx[b]] += float(w.get(a,1.0))
    tp=np.diag(cm); col=cm.sum(0); row=cm.sum(1)
    prec=np.divide(tp,col,out=np.zeros_like(tp),where=col>0)
    rec =np.divide(tp,row,out=np.zeros_like(tp),where=row>0)
    f1  =np.divide(2*prec*rec,prec+rec,out=np.zeros_like(tp),where=(prec+rec)>0)
    per = pd.DataFrame({"class":labs,"precision":prec,"recall":rec,"f1":f1,"support":row})
    return float(tp.sum()/cm.sum()), float(f1.mean()), per

def rare_recall(per):
    """Mean recall over classes below RARE_SHARE of the partition, excluding benign."""
    tot = per.support.sum()
    ben = [c for c in per["class"] if str(c).lower()=="benign"]
    r = per[(~per["class"].isin(ben)) & (per.support/tot < RARE_SHARE)]
    return float(r.recall.mean()) if len(r) else np.nan
print("protocol helpers ready")

## Experiment 5 — size-by-balance factorial on CIC-UNSW-NB15 $V_1$

The same four-arm design as the reported CIC-IDS2017 $V_3$ experiment, on the second source.
The small budget equals the matched class-balanced size of the strict panel on this cell
(14,229 rows); the large budget is 400,000. The balanced arms oversample the rarest classes with
replacement, and the realised size, unique-row count and oversampling factor are recorded.
Output: `exp5_size_by_balance_unsw.csv` (same schema as `exp1_size_by_balance.csv`).

In [ ]:
if RUN_2X2_UNSW:
    from xgboost import XGBClassifier
    CELL_WHICH, CELL_GROUPED, USE_PCA = "unsw", False, False   # CIC-UNSW-NB15 V1
    N_SMALL, N_LARGE = 14_229, 400_000   # matched balanced size on this cell

    base = load_base(CELL_WHICH, CELL_GROUPED)

    def natural(df, n, seed):
        return df.sample(min(n, len(df)), random_state=seed)

    def balanced_to(df, n_total, seed):
        """Class-balanced sample of about n_total records.

        A balanced sample drawn without replacement is bounded by the rarest
        class: on CIC-IDS2017 that is Heartbleed with a handful of training
        records, so a large balanced arm is impossible without oversampling and
        the 2x2 would degenerate into a 1x2. Classes with fewer records than the
        per-class quota are therefore oversampled with replacement, which is what
        a practitioner balancing a corpus would do. The realised size and the
        oversampling factor are recorded so the arm is never mistaken for an
        independent sample.
        """
        counts = df.Label.value_counts()
        per = max(1, n_total // len(counts))
        r = np.random.default_rng(seed); parts = []
        for _, p in df.groupby("Label", sort=True):
            idx = r.choice(len(p), per, replace=len(p) < per)
            parts.append(p.iloc[idx])
        return pd.concat(parts)

    RES = OUT/"exp5_size_by_balance_unsw.csv"
    EXPECTED = {"seed","arm","size","balance","n_train","n_unique_rows",
                "oversample_factor","min_class","max_class","accuracy","macro_f1",
                "rare_recall","fit_s"}
    rows, done = [], set()
    if RES.exists():
        prev = pd.read_csv(RES)
        if EXPECTED.issubset(prev.columns):          # resume
            rows = prev.to_dict("records")
            done = set(map(tuple, prev[["seed","arm"]].values))
            print(f"resuming: {len(done)} arms already complete")
        else:                                        # schema changed: start fresh
            RES.rename(RES.with_suffix(".csv.bak"))
            print(f"existing {RES.name} has an older schema "
                  f"(missing {sorted(EXPECTED - set(prev.columns))}); "
                  f"moved to {RES.name}.bak and starting fresh")

    for seed in SEEDS:
        tr, te, _ = split(base, seed)
        te_c, w = cap_test(te, seed)
        yte = te_c.Label.astype(str).reset_index(drop=True)
        arms = {"small_balanced": lambda: balanced_to(tr, N_SMALL, seed),
                "small_natural":  lambda: natural(tr, N_SMALL, seed),
                "large_balanced": lambda: balanced_to(tr, N_LARGE, seed),
                "large_natural":  lambda: natural(tr, N_LARGE, seed)}
        for arm, mk in arms.items():
            if (seed, arm) in done: continue
            sel = mk()
            xtr, xte = transform(sel, te_c, USE_PCA)
            ytr = sel.Label.astype(str).reset_index(drop=True)
            le = LabelEncoder().fit(ytr)
            set_seed(seed)
            m = XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1,
                              subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                              random_state=seed, n_jobs=-1, eval_metric="mlogloss")
            t0=time.time(); m.fit(xtr.to_numpy(), le.transform(ytr)); ft=time.time()-t0
            pred = le.inverse_transform(m.predict(xte.to_numpy()))
            acc, mf1, per = weighted_metrics(yte, pred, w)
            vc = sel.Label.value_counts()
            rows.append({"seed":seed,"arm":arm,
                         "size":"small" if arm.startswith("small") else "large",
                         "balance":"balanced" if arm.endswith("balanced") else "natural",
                         "n_train":len(sel),"n_unique_rows":sel.index.nunique(),
                         "oversample_factor":round(len(sel)/max(1,sel.index.nunique()),2),
                         "min_class":int(vc.min()),"max_class":int(vc.max()),
                         "accuracy":acc,"macro_f1":mf1,
                         "rare_recall":rare_recall(per),"fit_s":ft})
            pd.DataFrame(rows).to_csv(RES, index=False)
            print(f"  seed {seed} {arm:15} n={len(sel):>7,} acc={acc:.4f} "
                  f"rare_recall={rows[-1]['rare_recall']:.4f} ({ft:.0f}s)")

    E = pd.DataFrame(rows)
    print("\ncell means:")
    print(E.groupby(["size","balance"])[["n_train","n_unique_rows","oversample_factor",
                                         "accuracy","rare_recall"]].mean().round(4).to_string())
    if (E[E.balance=="balanced"].oversample_factor > 1.05).any():
        print("\nNote: the balanced arms oversample the rarest classes with replacement,")
        print("so their unique-row count is below their nominal size. Report both.")
    print("\nmain effects on rare-class recall, paired over seeds:")
    for fix in ("size","balance"):
        other = "balance" if fix=="size" else "size"
        for lvl in sorted(E[fix].unique()):
            g = E[E[fix]==lvl].pivot_table(index="seed", columns=other, values="rare_recall")
            if g.shape[1]==2:
                a,b = list(g.columns)
                d = (g[a]-g[b]).dropna().to_numpy()
                if len(d) < 2: continue
                lo,hi = ci(d)
                print(f"  holding {fix}={lvl:9} : {a} - {b} = {d.mean():+.4f} "
                      f"[{lo:+.4f},{hi:+.4f}] p={sf(d):.4f}")
    print("\nRead the 'holding size' rows for the class-balance effect and the")
    print("'holding balance' rows for the record-count effect. If balance is large at")
    print("both sizes and size small at both balances, Section 4.2 should describe a")
    print("class-balance effect and the confounding caveat can be removed.")
else:
    print("Experiment 5 disabled.")

## Experiment 6 — data-rich tree references on the Axis-2 evaluation rows

The matched 45-row anchor answers the controlled question; it cannot show whether a language
model is competitive with a realistically trained detector. This cell rebuilds the exact
Axis-2 split of the released campaign (same Kaggle table, same `three_way` logic, same seeds),
**verifies** that the re-derived evaluation labels equal the persisted `y_true` of every seed and
that the re-fitted 45-row anchor reproduces the persisted anchor accuracy, and then scores two
data-rich references on the identical evaluation rows and the persisted twelve columns:

* **development-rich**: in-context rows ∪ development partition (45 + ≈3,200 rows);
* **large natural**: in-context rows ∪ a natural-distribution sample of up to 400,000 rows
  drawn from every row that is neither in-context nor evaluation.

The in-context rows are included because Heartbleed (11 cleaned records) is exhausted by the
in-context and evaluation partitions: without them no data-rich reference could see that class.
Each reference uses the strong Random Forest and the XGBoost of the Axis-1 panel.
Outputs: `exp6_axis2_datarich.csv`, `exp6_axis2_datarich_perclass.csv`.

In [ ]:
if RUN_AXIS2_DATARICH:
    import kagglehub
    from sklearn.feature_selection import mutual_info_classif
    from sklearn.preprocessing import LabelEncoder
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
    from xgboost import XGBClassifier

    # --- data preparation, identical to axis2_six_model_campaign.ipynb (no subsampling) ----
    CIC_MAP = {"BENIGN":"BENIGN","Benign":"BENIGN","DDoS":"DDoS","DoS Hulk":"DoS",
               "DoS GoldenEye":"DoS","DoS slowloris":"DoS","DoS Slowhttptest":"DoS",
               "PortScan":"Port Scan","Portscan":"Port Scan","FTP-Patator":"Brute Force",
               "SSH-Patator":"Brute Force","Bot":"Bot","Infiltration":"Infiltration",
               "Heartbleed":"Heartbleed"}
    def group(v):
        s = re.sub(r"[^\x00-\x7F]+", " ", str(v)).replace("-", " ")
        s = re.sub(r"\s+", " ", s).strip()
        if "web attack" in s.lower(): return "Web Attack"
        return CIC_MAP.get(s, CIC_MAP.get(str(v).strip(), s))

    root = Path(kagglehub.dataset_download("dhoogla/cicids2017"))
    files = sorted(list(root.rglob("*.parquet")) + list(root.rglob("*.csv")))
    A2 = pd.concat([(pd.read_parquet(f) if f.suffix == ".parquet" else pd.read_csv(f, low_memory=True))
                    for f in files], ignore_index=True)
    A2.columns = [str(c).strip().replace(" ", "_") for c in A2.columns]
    lab = "ClassLabel" if "ClassLabel" in A2.columns else "Label"
    A2["Label"] = A2[lab].map(group)
    num = A2.select_dtypes(include=[np.number]).columns.tolist()
    A2 = A2[num + ["Label"]].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    print(f"{len(A2):,} rows, {A2.Label.nunique()} classes")

    K_FEATURES, K_SHOTS, N_TEST_PER_CLASS, N_DEV_PER_CLASS, N_LARGE = 12, 5, 30, 400, 400_000

    def three_way_main(d, seed):
        rng = np.random.default_rng(seed)
        dev, icl, test = [], [], []
        for _, part in d.groupby("Label", sort=True):
            part = part.iloc[rng.permutation(len(part))]
            n_dev = min(N_DEV_PER_CLASS, max(0, len(part) - K_SHOTS - N_TEST_PER_CLASS))
            rest = len(part) - n_dev
            n_icl = min(K_SHOTS, rest); n_test = min(N_TEST_PER_CLASS, max(0, rest - n_icl))
            dev.append(part.iloc[:n_dev]); icl.append(part.iloc[n_dev:n_dev+n_icl])
            test.append(part.iloc[n_dev+n_icl:n_dev+n_icl+n_test])
        cat = lambda ps: pd.concat([p for p in ps if len(p)])   # keep original index for exclusion
        return cat(dev), cat(icl), cat(test)

    def mi_features(d, k, seed):
        x = d.drop(columns=["Label"], errors="ignore").select_dtypes(include=[np.number])
        y = LabelEncoder().fit_transform(d["Label"].astype(str))
        return list(x.columns[np.argsort(mutual_info_classif(x, y, random_state=seed))[::-1][:k]])

    # --- persisted record of the released campaign (for verification and columns) --------
    REPO = Path(REPO_DIR).resolve()
    RAW = REPO/"results"/"axis2"/"raw_generations_original_six_model_campaign.jsonl"
    RESCSV = REPO/"results"/"axis2"/"results_original_six_model_campaign.csv"
    assert RAW.exists() and RESCSV.exists(), "clone the released repository into REPO_DIR first"
    raw = [json.loads(l) for l in RAW.read_text().splitlines() if l.strip()]
    persisted = {}
    for r in raw:
        persisted.setdefault(int(r["seed"]), {"y_true": [str(v) for v in r["y_true"]],
                                              "features": list(r["features"])})
    anchor_ref = (pd.read_csv(RESCSV).query("model == 'RandomForest'")
                    .drop_duplicates("seed").set_index("seed"))

    def make_xgb(seed):
        return XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
                             colsample_bytree=0.9, tree_method="hist", random_state=seed,
                             n_jobs=-1, eval_metric="mlogloss")
    def make_rf(seed):
        return RandomForestClassifier(n_estimators=200, max_features="sqrt",
                                      random_state=seed, n_jobs=-1)

    RES = OUT/"exp6_axis2_datarich.csv"
    rows = pd.read_csv(RES).to_dict("records") if RES.exists() else []
    done = {(r["seed"], r["reference"], r["learner"]) for r in rows}
    PC = OUT/"exp6_axis2_datarich_perclass.csv"
    pc_rows = pd.read_csv(PC).to_dict("records") if PC.exists() else []

    for seed in SEEDS:
        dev, icl, test = three_way_main(A2, seed)
        y_true = test["Label"].astype(str).to_numpy()
        # --- verification against the persisted campaign ---------------------------------
        if seed in persisted:
            same_rows = list(y_true) == persisted[seed]["y_true"]
            feats = persisted[seed]["features"]
            feats_local = mi_features(dev, K_FEATURES, seed)
            print(f"seed {seed}: rows={'MATCH' if same_rows else 'MISMATCH'} "
                  f"features={'match' if feats == feats_local else 'differ (persisted columns used)'}")
            assert same_rows, ("The re-derived evaluation labels differ from the persisted campaign; "
                               "the data preparation is not identical. Do not proceed.")
        else:
            feats = mi_features(dev, K_FEATURES, seed)
        labels = sorted(set(y_true))

        # --- anchor reproduction check ---------------------------------------------------
        rf0 = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
        rf0.fit(icl[feats].to_numpy(), icl["Label"].astype(str).to_numpy())
        acc0 = accuracy_score(y_true, rf0.predict(test[feats].to_numpy()))
        if seed in anchor_ref.index:
            print(f"  anchor accuracy re-fitted {acc0:.4f} vs persisted {anchor_ref.loc[seed,'accuracy']:.4f}")

        # --- data-rich fitting sets -------------------------------------------------------
        excluded = set(icl.index) | set(test.index)
        pool = A2.drop(index=list(excluded))
        rng = np.random.default_rng(seed)
        large = pool.iloc[rng.choice(len(pool), min(N_LARGE, len(pool)), replace=False)]
        fits = {"development-rich": pd.concat([icl, dev]),
                "large natural":    pd.concat([icl, large])}
        for ref, F in fits.items():
            for learner, mk in [("strong RF", make_rf), ("XGBoost", make_xgb)]:
                if (seed, ref, learner) in done: continue
                ytr = F["Label"].astype(str).to_numpy()
                le = LabelEncoder().fit(ytr)
                m = mk(seed); t0 = time.time()
                if learner == "XGBoost":
                    m.fit(F[feats].to_numpy(), le.transform(ytr))
                    pred = le.inverse_transform(m.predict(test[feats].to_numpy()))
                else:
                    m.fit(F[feats].to_numpy(), ytr)
                    pred = m.predict(test[feats].to_numpy())
                ft = time.time()-t0
                p_, r_, f_, s_ = precision_recall_fscore_support(y_true, pred, labels=labels,
                                                                 zero_division=0)
                rows.append({"seed": seed, "reference": ref, "learner": learner, "n_fit": len(F),
                             "n_fit_unique": F.index.nunique(), "n_eval": len(test),
                             "accuracy": accuracy_score(y_true, pred),
                             "macro_f1": f1_score(y_true, pred, labels=labels, average="macro",
                                                  zero_division=0),
                             "anchor_accuracy_refit": acc0, "fit_s": ft})
                for lb, pv, rv, fv, sv in zip(labels, p_, r_, f_, s_):
                    pc_rows.append({"seed": seed, "reference": ref, "learner": learner, "class": lb,
                                    "precision": float(pv), "recall": float(rv), "f1": float(fv),
                                    "support": int(sv)})
                pd.DataFrame(rows).to_csv(RES, index=False)
                pd.DataFrame(pc_rows).to_csv(PC, index=False)
                print(f"  {ref:18} {learner:9} n_fit={len(F):>7,} acc={rows[-1]['accuracy']:.4f} "
                      f"macroF1={rows[-1]['macro_f1']:.4f} ({ft:.0f}s)")
    E6 = pd.DataFrame(rows)
    print("\nmeans over seeds:")
    print(E6.groupby(["reference","learner"])[["n_fit","accuracy","macro_f1"]]
            .mean().round(4).to_string())
else:
    print("Experiment 6 disabled.")

## Experiment 4b — ManyClass sensitivity, seeds 3–9

Extends the released three-seed ManyClass experiment to ten seeds under the identical
protocol (CIC-IDS2017 $V_1$, matched balanced set of 2,000 rows per class, seed-specific
evaluation cap). The cell resumes from `exp4_manyclass.csv`; if the file is absent it copies
the released three-seed record so that seeds 0–2 are not recomputed. Needs a GPU and `HF_TOKEN`.

**Consistency note.** Seeds 0–2 are resumed from the released record. The vendored loader concatenates the eight raw CSVs in the fixed capture-day order that `_prefetch_cic` writes them, which reproduces the released environment; if the appended seeds nevertheless look discontinuous against seeds 0–2 (a jump in `rare_recall` well outside the seed spread), delete the copied `exp4_manyclass.csv` and rerun all ten seeds so the table is internally consistent.

In [ ]:
!pip install tabpfn

In [ ]:
# ============================================================
# PATCH — remove bad constructor patch and force GPU correctly
# ============================================================

import importlib
import torch

assert torch.cuda.is_available(), "GPU disappeared."

# ------------------------------------------------------------------
# Undo the previous monkey-patch completely.
# ------------------------------------------------------------------

tabpfn_module = importlib.import_module("src.models.trainable.tabpfn")
tabpfn_module = importlib.reload(tabpfn_module)

mcfg = importlib.import_module("src.models.config")

# ------------------------------------------------------------------
# These are the parameters consumed by TabPFNClassifier itself.
# ------------------------------------------------------------------

mcfg.TABPFN_CONFIG["device"] = "cuda"
mcfg.TABPFN_CONFIG["ignore_pretraining_limits"] = True

print("[patch] TABPFN_CONFIG =", {
    "device": mcfg.TABPFN_CONFIG["device"],
    "ignore_pretraining_limits":
        mcfg.TABPFN_CONFIG["ignore_pretraining_limits"],
})

# ------------------------------------------------------------------
# Patch AFTER construction, not BEFORE.
# ------------------------------------------------------------------

_original_init = tabpfn_module.TabPFNModel.__init__

def _gpu_init(self, *args, **kwargs):

    _original_init(self, *args, **kwargs)

    if hasattr(self.model, "set_params"):

        params = self.model.get_params(deep=False)

        updates = {}

        if "device" in params:
            updates["device"] = "cuda"

        if "ignore_pretraining_limits" in params:
            updates["ignore_pretraining_limits"] = True

        if updates:
            self.model.set_params(**updates)

    # sanity check

    try:
        p = self.model.get_params(deep=False)

        print(
            "[patch] estimator device =",
            p.get("device", "<missing>"),
        )

    except Exception:
        pass


tabpfn_module.TabPFNModel.__init__ = _gpu_init

print("[patch] constructor restored correctly")
print("[patch] GPU will be configured on the underlying estimator")

In [ ]:
# ============================================================
# PATCH — restore missing TabPFN 6.0.6 column-embedding asset
# ============================================================
#
# TabPFN 6.0.6 expects:
#   tabpfn/architectures/shared/tabpfn_col_embedding.pt
#
# The 6.0.6 wheel can omit this file because its package-data entry points
# to architectures.base instead of architectures.shared.
#
# This patch downloads the exact official asset from the immutable v6.0.6
# release commit, validates that it is a readable PyTorch tensor payload,
# and places it at the path expected by the installed package.

from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import importlib.util
import io
import os
import sys
import tempfile
import zipfile
from pathlib import Path

import requests
import torch


# ------------------------------------------------------------------
# 1. Verify that this is the package version for which the patch is
#    intended.
# ------------------------------------------------------------------

installed_version = importlib.metadata.version("tabpfn")

if installed_version != "6.0.6":
    raise RuntimeError(
        "This patch is specifically for tabpfn==6.0.6, but the active "
        f"environment contains tabpfn=={installed_version}."
    )


# ------------------------------------------------------------------
# 2. Locate the exact installed package directory.
# ------------------------------------------------------------------

shared_spec = importlib.util.find_spec("tabpfn.architectures.shared")

if shared_spec is None or not shared_spec.submodule_search_locations:
    raise RuntimeError(
        "Could not locate the installed "
        "tabpfn.architectures.shared package."
    )

shared_dir = Path(
    next(iter(shared_spec.submodule_search_locations))
).resolve()

shared_dir.mkdir(parents=True, exist_ok=True)

destination = shared_dir / "tabpfn_col_embedding.pt"


# ------------------------------------------------------------------
# 3. Exact official source revision for TabPFN v6.0.6.
# ------------------------------------------------------------------

TABPFN_V606_COMMIT = (
    "4103e892d8c3059183ac7c507b923924ac9b1091"
)

# Try both source locations. The runtime expects "shared"; the package-data
# declaration in this release refers to "base".
source_relatives = [
    "src/tabpfn/architectures/shared/tabpfn_col_embedding.pt",
    "src/tabpfn/architectures/base/tabpfn_col_embedding.pt",
]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def collect_tensors(value):
    """Recursively collect tensors from a torch.load payload."""

    tensors = []

    if torch.is_tensor(value):
        tensors.append(value)

    elif isinstance(value, dict):
        for item in value.values():
            tensors.extend(collect_tensors(item))

    elif isinstance(value, (list, tuple)):
        for item in value:
            tensors.extend(collect_tensors(item))

    return tensors


def validate_embedding_file(path: Path):
    """Reject HTML, Git-LFS pointers, truncated files, or invalid payloads."""

    if not path.is_file():
        raise FileNotFoundError(path)

    size = path.stat().st_size

    if size < 512:
        raise RuntimeError(
            f"Downloaded embedding file is unexpectedly small: {size} bytes"
        )

    head = path.read_bytes()[:256].lstrip()

    if head.startswith((b"<", b"<!DOCTYPE", b"<?xml")):
        raise RuntimeError(
            "The embedding download returned an HTML/XML response."
        )

    if head.startswith(b"version https://git-lfs.github.com/spec"):
        raise RuntimeError(
            "The download returned a Git-LFS pointer rather than the asset."
        )

    try:
        payload = torch.load(
            path,
            map_location="cpu",
            weights_only=True,
        )
    except TypeError:
        # Compatibility with older supported PyTorch releases.
        payload = torch.load(
            path,
            map_location="cpu",
        )

    tensors = collect_tensors(payload)

    if not tensors:
        raise RuntimeError(
            "The downloaded file is readable by torch.load, but contains "
            "no tensor payload."
        )

    total_values = sum(int(t.numel()) for t in tensors)

    if total_values == 0:
        raise RuntimeError(
            "The downloaded embedding tensor payload is empty."
        )

    summary = {
        "bytes": size,
        "sha256": sha256_file(path),
        "tensor_count": len(tensors),
        "total_values": total_values,
        "first_shape": tuple(tensors[0].shape),
        "first_dtype": str(tensors[0].dtype),
    }

    return summary


# ------------------------------------------------------------------
# 4. Keep a valid existing asset; otherwise download it.
# ------------------------------------------------------------------

existing_valid = False

if destination.exists():
    try:
        asset_summary = validate_embedding_file(destination)
        existing_valid = True
        print("[patch] existing embedding asset is already valid")
    except Exception as exc:
        invalid_backup = destination.with_suffix(
            ".pt.invalid"
        )
        invalid_backup.unlink(missing_ok=True)
        destination.replace(invalid_backup)

        print(
            "[patch] moved invalid existing asset to:",
            invalid_backup,
        )
        print("[patch] validation failure was:", repr(exc))


download_source = None

if not existing_valid:
    last_error = None

    # First try the raw and media endpoints at the immutable release commit.
    candidate_urls = []

    for relative in source_relatives:
        candidate_urls.extend(
            [
                (
                    "https://raw.githubusercontent.com/"
                    "PriorLabs/TabPFN/"
                    f"{TABPFN_V606_COMMIT}/{relative}"
                ),
                (
                    "https://media.githubusercontent.com/media/"
                    "PriorLabs/TabPFN/"
                    f"{TABPFN_V606_COMMIT}/{relative}"
                ),
            ]
        )

    for url in candidate_urls:
        tmp_path = destination.with_suffix(".pt.downloading")
        tmp_path.unlink(missing_ok=True)

        try:
            print("[patch] trying:", url)

            with requests.get(
                url,
                stream=True,
                timeout=(30, 180),
                headers={
                    "User-Agent": "round4-tabpfn-6.0.6-asset-recovery"
                },
            ) as response:
                response.raise_for_status()

                with tmp_path.open("wb") as handle:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            handle.write(chunk)

            asset_summary = validate_embedding_file(tmp_path)
            os.replace(tmp_path, destination)
            download_source = url
            break

        except Exception as exc:
            last_error = exc
            tmp_path.unlink(missing_ok=True)

    # Fall back to the complete GitHub source archive if direct raw access
    # is unavailable.
    if download_source is None:
        archive_url = (
            "https://codeload.github.com/PriorLabs/TabPFN/zip/"
            f"{TABPFN_V606_COMMIT}"
        )

        print("[patch] direct asset download failed; trying source archive")

        try:
            response = requests.get(
                archive_url,
                timeout=(30, 300),
                headers={
                    "User-Agent": "round4-tabpfn-6.0.6-asset-recovery"
                },
            )
            response.raise_for_status()

            with zipfile.ZipFile(
                io.BytesIO(response.content)
            ) as archive:
                members = []

                for member in archive.namelist():
                    normalized = member.replace("\\", "/")

                    if any(
                        normalized.endswith("/" + relative)
                        for relative in source_relatives
                    ):
                        members.append(member)

                if not members:
                    raise FileNotFoundError(
                        "The official v6.0.6 source archive did not "
                        "contain tabpfn_col_embedding.pt."
                    )

                data = archive.read(members[0])

            tmp_path = destination.with_suffix(".pt.downloading")
            tmp_path.write_bytes(data)

            asset_summary = validate_embedding_file(tmp_path)
            os.replace(tmp_path, destination)
            download_source = archive_url

        except Exception as archive_error:
            raise RuntimeError(
                "Could not recover the official TabPFN 6.0.6 "
                "column-embedding asset."
            ) from archive_error

    # Validate once more after the atomic move.
    asset_summary = validate_embedding_file(destination)


# ------------------------------------------------------------------
# 5. Release any failed model object/GPU allocation from the preceding
#    interrupted prediction.
# ------------------------------------------------------------------

if "m" in globals():
    try:
        del globals()["m"]
    except Exception:
        pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------------
# 6. Final report.
# ------------------------------------------------------------------

print("\n[patch] TabPFN 6.0.6 embedding asset ready")
print("  destination :", destination)
print("  source      :", download_source or "existing valid file")
print("  bytes       :", f"{asset_summary['bytes']:,}")
print("  sha256      :", asset_summary["sha256"])
print("  tensors     :", asset_summary["tensor_count"])
print("  total values:", f"{asset_summary['total_values']:,}")
print("  first shape :", asset_summary["first_shape"])
print("  first dtype :", asset_summary["first_dtype"])

assert destination.is_file()
assert destination.stat().st_size == asset_summary["bytes"]

print("\n[patch] PASS — rerun Experiment 4 unchanged")

In [ ]:
# ============================================================
# PATCH — restore the exact Round-2 TabPFN wrapper for Experiment 4b
# Run after the repository/loader setup and before Experiment 4.
# Then rerun Experiment 4 unchanged.
# ============================================================

import importlib
import os
import subprocess
import sys
from pathlib import Path

# The original combined installation failed on tabicl==0.1.3.
# Install only the pinned packages required by this experiment.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tabpfn==6.0.6",
        "tabpfn-extensions==0.2.1",
        "tabpfn-common-utils==0.2.10",
    ],
    check=True,
)

assert os.environ.get("HF_TOKEN"), (
    "HF_TOKEN is missing. Add it to Colab secrets and rerun "
    "the secrets/setup cell."
)

repo = Path(REPO_DIR).resolve()
assert (repo / ".git").is_dir(), (
    f"Expected a complete Git clone at {repo}"
)

# This revision contains the model wrapper used in the preceding round.
# Only the four files required by TabPFN are recovered. The broad historical
# __init__.py files are deliberately not restored because they import unrelated
# optional models such as TabICL, TabSTAR, TabDPT, and TabNet.
source_commit = "782486cb947852ece7627345e5dc6a83582bbb67"

needed_files = [
    "src/models/config.py",
    "src/models/_base.py",
    "src/models/trainable/_base.py",
    "src/models/trainable/tabpfn.py",
]

# Confirm that the source revision exists locally. A normal full clone already
# contains it; the fetch is only a fallback for an incomplete clone.
probe = subprocess.run(
    [
        "git",
        "-C",
        str(repo),
        "cat-file",
        "-e",
        f"{source_commit}^{{commit}}",
    ],
    capture_output=True,
)

if probe.returncode != 0:
    print("[patch] historical source revision not local; fetching it")
    subprocess.run(
        [
            "git",
            "-C",
            str(repo),
            "fetch",
            "origin",
            source_commit,
        ],
        check=True,
    )

# Recover the exact historical files without switching the repository branch.
for relative_path in needed_files:
    content = subprocess.check_output(
        [
            "git",
            "-C",
            str(repo),
            "show",
            f"{source_commit}:{relative_path}",
        ]
    )

    target = repo / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)

# Minimal package initializers: expose only the model required here.
(repo / "src/models/__init__.py").write_text(
    "from .trainable.tabpfn import TabPFNModel\n",
    encoding="utf-8",
)

(repo / "src/models/trainable/__init__.py").write_text(
    "from .tabpfn import TabPFNModel\n",
    encoding="utf-8",
)

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

# Reapply the TabPFN compatibility shim now that TabPFN has actually been
# installed. The earlier setup cell may have skipped this because the import
# was unavailable at that point.
import tabpfn.inference_config as _tabpfn_inference_config

if not hasattr(
    _tabpfn_inference_config,
    "default_classifier_preprocessor_configs",
):
    for candidate_name in (
        "v2_5_classifier_preprocessor_configs",
        "v2_classifier_preprocessor_configs",
    ):
        if hasattr(_tabpfn_inference_config, candidate_name):
            _tabpfn_inference_config.default_classifier_preprocessor_configs = (
                getattr(_tabpfn_inference_config, candidate_name)
            )
            break

assert hasattr(
    _tabpfn_inference_config,
    "default_classifier_preprocessor_configs",
), "No compatible TabPFN classifier-preprocessor factory was found."

# Compatibility with huggingface_hub releases that no longer expose HfFolder.
import huggingface_hub as _huggingface_hub

if not hasattr(_huggingface_hub, "HfFolder"):

    class HfFolder:
        path_token = str(
            Path.home() / ".cache" / "huggingface" / "token"
        )

        @classmethod
        def get_token(cls):
            try:
                return _huggingface_hub.get_token()
            except Exception:
                return None

        @classmethod
        def save_token(cls, token):
            path = Path(cls.path_token)
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(token or "", encoding="utf-8")

        @classmethod
        def delete_token(cls):
            Path(cls.path_token).unlink(missing_ok=True)

    _huggingface_hub.HfFolder = HfFolder

# Remove stale or partially imported source modules from the failed attempt.
# The notebook's vendored data loaders are ordinary objects and are unaffected.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]


def _import_with_stubs(target):
    """
    Compatibility helper expected by the existing Experiment-4 cell.

    No optional-package stubs are required because the restored package
    initializers expose only TabPFNModel.
    """
    return importlib.import_module(target), []


# Import-only smoke test. This does not construct TabPFN or download weights.
_models, _stubbed_models = _import_with_stubs("src.models")

_mcfg = importlib.import_module("src.models.config")

_TabPFNModel = importlib.import_module(
    "src.models.trainable.tabpfn"
).TabPFNModel

assert _mcfg.TABPFN_CONFIG["n_estimators"] == 4
assert _mcfg.TABPFN_CONFIG["random_state"] == 0
assert _mcfg.TABPFN_MANY_CLASS_CONFIG["alphabet_size"] == 5
assert _mcfg.TABPFN_MANY_CLASS_CONFIG["n_estimators_redundancy"] == 4
assert _mcfg.TABPFN_EXPERT_CONFIG["MAX_NUMBER_OF_CLASSES"] == 10

print(
    "[patch] exact Round-2 TabPFN/ManyClass wrapper "
    "restored successfully"
)
print(
    "[patch] rerun Experiment 4 unchanged; "
    "the separate '!pip install tabstar' cell is not needed"
)

In [ ]:
import os; os.environ["TABPFN_TOKEN"] = userdata.get('TABPFN_TOKEN')

In [ ]:
# ============================================================
# PATCH — allow large TabPFN inference
# ============================================================

import os
import torch

# Allow the safety limit to be bypassed if CPU must be used.
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

# Detect runtime.
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"[patch] GPU detected: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("[patch] WARNING: running TabPFN on CPU.")
    print("[patch] Prediction on ~45k evaluation rows can be very slow.")

# -----------------------------------------------------------------
# Patch the restored wrapper so it forwards the device explicitly.
# -----------------------------------------------------------------

import importlib

_tabpfn = importlib.import_module("src.models.trainable.tabpfn")

_old_init = _tabpfn.TabPFNModel.__init__

def _patched_init(self, *args, **kwargs):

    kwargs.setdefault("device", DEVICE)

    # If CPU, bypass the TabPFN pretraining safeguard.
    kwargs.setdefault("ignore_pretraining_limits", True)

    return _old_init(self, *args, **kwargs)

_tabpfn.TabPFNModel.__init__ = _patched_init

print(f"[patch] TabPFNModel will run on {DEVICE}.")

In [ ]:
# ============================================================
#  EXPERIMENT 4 — ManyClass sensitivity  (corrected)
# ============================================================
if RUN_MANYCLASS_10:
    import importlib

    _models, _stubbed_m = _import_with_stubs("src.models")
    mcfg        = importlib.import_module("src.models.config")
    TabPFNModel = importlib.import_module("src.models.trainable.tabpfn").TabPFNModel
    print(f"src.models OK" + (f"; placeholders: {_stubbed_m}" if _stubbed_m else ""))

    base  = load_base("cic", False)          # CIC-IDS2017 V1, 15 classes
    saved = dict(mcfg.TABPFN_MANY_CLASS_CONFIG)

    RES  = OUT/"exp4_manyclass.csv"
    _rel = Path(REPO_DIR)/"results"/"sensitivity"/"exp4_manyclass.csv"
    if not RES.exists() and _rel.exists():
        shutil.copy2(_rel, RES); print("copied released three-seed record")
    rows = pd.read_csv(RES).to_dict("records") if RES.is_file() else []
    done = {(r["alphabet_size"], r["n_estimators_redundancy"], r["seed"]) for r in rows}
    if done: print(f"resuming: {len(done)} of {4*len(SEEDS)} runs already complete")

    try:
        for alphabet in (5, 10):
            for redundancy in (1, 4):
                for seed in SEEDS:
                    if (alphabet, redundancy, seed) in done: continue
                    # Set the wrapper configuration BEFORE constructing the model:
                    # TabPFNModel reads it at construction, not at fit.
                    mcfg.TABPFN_MANY_CLASS_CONFIG.update(
                        alphabet_size=alphabet, n_estimators_redundancy=redundancy,
                        random_state=seed)
                    mcfg.TABPFN_CONFIG["random_state"] = seed

                    tr, te, _ = split(base, seed)
                    te_c, w   = cap_test(te, seed)
                    sel  = balanced(tr, 2000, seed)
                    xtr, xte = transform(sel, te_c, False)
                    ytr = sel.Label.astype(str).reset_index(drop=True)
                    yte = te_c.Label.astype(str).reset_index(drop=True)

                    set_seed(seed)
                    m = TabPFNModel(name=f"tabpfn_a{alphabet}_r{redundancy}_s{seed}",
                                    version="2.5", use_default_config=False)
                    t0 = time.time(); m.fit(xtr.to_numpy(), ytr.to_numpy()); ft = time.time()-t0
                    pred = np.asarray(m.predict(xte.to_numpy()))
                    acc, mf1, per = weighted_metrics(yte, pred, w)

                    n_classes = int(pd.Series(ytr).nunique())
                    rows.append({"alphabet_size":alphabet,"n_estimators_redundancy":redundancy,
                                 "seed":seed,"n_classes":n_classes,
                                 "wrapper_engaged":n_classes > 10,
                                 "accuracy":acc,"macro_f1":mf1,
                                 "rare_recall":rare_recall(per),"fit_s":ft})
                    pd.DataFrame(rows).to_csv(RES, index=False)
                    print(f"  a={alphabet} r={redundancy} seed={seed} acc={acc:.4f} "
                          f"rare_recall={rows[-1]['rare_recall']:.4f} ({ft:.0f}s)")
                    del m; gc.collect()
    finally:
        mcfg.TABPFN_MANY_CLASS_CONFIG.clear()
        mcfg.TABPFN_MANY_CLASS_CONFIG.update(saved)

    MC = pd.DataFrame(rows)
    # If the wrapper never engaged, the 2x2 measures nothing and must not be read
    # as evidence that the setting is immaterial.
    if not MC.wrapper_engaged.all():
        print(f"\nWARNING: the wrapper engaged on {int(MC.wrapper_engaged.sum())} of "
              f"{len(MC)} runs. It activates above ten classes; this cell fits "
              f"{int(MC.n_classes.iloc[0])}. Results below are not a sensitivity test.")
    print("\n" + MC.groupby(["alphabet_size","n_estimators_redundancy"])
             [["accuracy","macro_f1","rare_recall"]].mean().round(4).to_string())
    g = MC.groupby(["alphabet_size","n_estimators_redundancy"]).rare_recall.mean()
    print(f"\nrare-class recall spread across settings: {g.max()-g.min():.4f}")
    print("Compare against the smallest margin reported in Table 6. If the spread is")
    print("below it, the wrapper is not load-bearing and Section 6.4 can say so with")
    print("evidence rather than by disclosure alone.")
else:
    print("Experiment 4b disabled.")

## Experiment 7 — six-model Axis-2 panel on CIC-UNSW-NB15 $V_1$

The identical Axis-2 protocol applied to a second source: up to 400 development, 5 in-context
and 30 evaluation rows per class (ten classes → 300 evaluation rows and a 50-row anchor per
seed), twelve mutual-information features selected on development rows only, deterministic
single vote, one label-only retry, closed-label scoring, and ten paired seeds. The frozen
configuration registry of the main campaign is re-verified on five blind probes per model
(all must parse and terminate with `stop`); a model whose frozen configuration fails falls
back through the same configuration ladder as the original Stage A.

**Cost.** The main campaign cost USD 218.34 for 14,760 scored rows. This variant scores
18,000 rows with ~10% longer prompts, so the full panel is ≈ USD 270–320 at the same prices.
Restrict `SECOND_VARIANT_MODELS` / `SECOND_VARIANT_SEEDS` in the switches cell to reduce it.
Outputs: `axis2_second_variant/results.csv`, `raw_generations.jsonl`, `stage_a_probe.csv`,
`stage_a_chosen_config.json`.

In [ ]:
if RUN_AXIS2_SECOND:
    from openai import OpenAI
    from sklearn.feature_selection import mutual_info_classif
    from sklearn.preprocessing import LabelEncoder
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score

    client = OpenAI(base_url="https://api.studio.nebius.com/v1/",
                    api_key=os.environ["NEBIUS_API_KEY"])
    OUT7 = OUT/"axis2_second_variant"; OUT7.mkdir(exist_ok=True)

    TARGETS = {"GLM-5.2": ["glm-5.2","glm5.2"], "DeepSeek-V4-Pro": ["deepseek-v4-pro","deepseekv4pro"],
               "Kimi-K3": ["kimi-k3","kimik3"], "MiniMax-M3": ["minimax-m3","minimaxm3"],
               "Qwen3.5-397B-A17B": ["qwen3.5-397b","397b-a17b"],
               "Nemotron-3-Ultra-550b-a55b": ["nemotron-3-ultra","550b-a55b"]}
    FROZEN = {"GLM-5.2": "plain_4096", "DeepSeek-V4-Pro": "plain_512", "Kimi-K3": "plain_4096",
              "MiniMax-M3": "plain_4096", "Qwen3.5-397B-A17B": "nothink_template",
              "Nemotron-3-Ultra-550b-a55b": "nothink_template"}
    PRICES = {"GLM-5.2": (1.40, 4.40), "DeepSeek-V4-Pro": (1.75, 3.50), "Kimi-K3": (3.00, 15.00),
              "MiniMax-M3": (0.30, 1.20), "Qwen3.5-397B-A17B": (0.60, 3.60),
              "Nemotron-3-Ultra-550b-a55b": (1.00, 3.00)}  # USD per 1M tokens (in, out), main campaign
    CONFIGS = [
        {"tag": "plain_512",        "max_tokens": 512,  "extra": None,  "prefill": False},
        {"tag": "plain_4096",       "max_tokens": 4096, "extra": None,  "prefill": False},
        {"tag": "nothink_template", "max_tokens": 4096, "prefill": False,
         "extra": {"chat_template_kwargs": {"thinking": False, "enable_thinking": False}}},
        {"tag": "nothink_effort",   "max_tokens": 4096, "prefill": False,
         "extra": {"reasoning_effort": "none"}},
        {"tag": "prefill_4096",     "max_tokens": 4096, "extra": None,  "prefill": True},
    ]
    CFG = {c["tag"]: c for c in CONFIGS}
    available = sorted(m.id for m in client.models.list().data)
    norm = lambda s: re.sub(r"[^a-z0-9.]", "", s.lower())
    RESOLVED = {}
    for name in SECOND_VARIANT_MODELS:
        hit = next((m for m in available if any(norm(p) in norm(m) for p in TARGETS[name])), None)
        if hit: RESOLVED[name] = hit
        print(f"  {'OK ' if hit else '-- '} {name:<28} {hit or 'not available'}")
    assert RESOLVED

    # --- second variant table: CIC-UNSW-NB15 V1 through the released loader --------------
    U = load_base("unsw", False)
    LABELS = sorted(U.Label.astype(str).unique())
    print(f"CIC-UNSW-NB15 V1: {len(U):,} rows, classes: {LABELS}")
    K_FEATURES, K_SHOTS, N_TEST_PER_CLASS, N_DEV_PER_CLASS = 12, 5, 30, 400

    def three_way_u(d, seed):
        rng = np.random.default_rng(seed); dev, icl, test = [], [], []
        for _, part in d.groupby("Label", sort=True):
            part = part.iloc[rng.permutation(len(part))]
            n_dev = min(N_DEV_PER_CLASS, max(0, len(part)-K_SHOTS-N_TEST_PER_CLASS))
            rest = len(part)-n_dev
            n_icl = min(K_SHOTS, rest); n_test = min(N_TEST_PER_CLASS, max(0, rest-n_icl))
            dev.append(part.iloc[:n_dev]); icl.append(part.iloc[n_dev:n_dev+n_icl])
            test.append(part.iloc[n_dev+n_icl:n_dev+n_icl+n_test])
        cat = lambda ps: pd.concat([p for p in ps if len(p)], ignore_index=True)
        return cat(dev), cat(icl), cat(test)
    def mi_features(d, k, seed):
        x = d.drop(columns=["Label"], errors="ignore").select_dtypes(include=[np.number])
        y = LabelEncoder().fit_transform(d["Label"].astype(str))
        return list(x.columns[np.argsort(mutual_info_classif(x, y, random_state=seed))[::-1][:k]])
    def build_prompt(icl, row, features):
        shots = "\n".join(json.dumps({**{f: round(float(r[f]), 4) for f in features},
                                      "label": str(r["Label"])})
                          for _, r in icl.iterrows())
        system = ("You are a network intrusion detection classifier. "
                  f"Choose exactly one label from this list: {', '.join(LABELS)}. "
                  "Respond with the label only, on its own line, with no explanation.")
        return system, f"{shots}\n" + json.dumps({f: round(float(row[f]), 4) for f in features})
    def parse_label(text):
        if not text: return None
        t = re.sub(r"<(think|thinking|reasoning)>.*?</\1>", " ", text, flags=re.S|re.I)
        t = re.sub(r"^.*</(think|thinking|reasoning)>", " ", t, flags=re.S|re.I)
        for line in [l.strip(" \t\"'`*.-:") for l in t.strip().splitlines()[::-1] if l.strip()]:
            for lab in sorted(LABELS, key=len, reverse=True):
                if line.lower() == lab.lower(): return lab
            for lab in sorted(LABELS, key=len, reverse=True):
                if lab.lower() in line.lower(): return lab
        return None
    def call(model_id, system, user, cfg, temperature=0.0):
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        if cfg.get("prefill"): msgs.append({"role": "assistant", "content": "Label:"})
        kw = dict(model=model_id, messages=msgs, max_tokens=cfg["max_tokens"],
                  temperature=temperature, timeout=300)
        if cfg.get("extra"): kw["extra_body"] = cfg["extra"]
        r = client.chat.completions.create(**kw); ch = r.choices[0]; u = getattr(r, "usage", None)
        return ((ch.message.content or ""), getattr(ch, "finish_reason", "?"),
                getattr(u, "prompt_tokens", 0) or 0, getattr(u, "completion_tokens", 0) or 0)

    # --- Stage A: re-verify the frozen registry on five blind probes (calibration seed 1000)
    PROBE_ROWS = 5
    devP, iclP, testP = three_way_u(U, 1000)   # calibration draw, never used for evaluation
    featsP = mi_features(devP, K_FEATURES, 1000)
    MODEL_CONFIG, probe_rows = {}, []
    for name, mid in RESOLVED.items():
        ladder = [CFG[FROZEN[name]]] + [c for c in CONFIGS if c["tag"] != FROZEN[name]]
        chosen = None
        for cfg in ladder:
            hits, clean, fr, preview, err = 0, 0, "", "", ""
            try:
                for j in range(PROBE_ROWS):
                    s_, u_ = build_prompt(iclP, testP.iloc[j], featsP)
                    txt, fr, ti, to = call(mid, s_, u_, cfg)
                    preview = preview or txt.strip().replace("\n", " ")[:110]
                    hits += int(parse_label(txt) is not None); clean += int(fr == "stop")
            except Exception as e:
                err = f"{type(e).__name__}: {str(e).splitlines()[0][:90]}"
            probe_rows.append({"model": name, "config": cfg["tag"], "parsed": hits, "clean": clean,
                               "of": PROBE_ROWS, "finish_reason": fr, "preview": preview,
                               "error": err, "frozen": cfg["tag"] == FROZEN[name]})
            print(f"  {name:<28} {cfg['tag']:<18} parsed={hits}/{PROBE_ROWS} "
                  f"clean={clean}/{PROBE_ROWS} {err[:50]}")
            if not err and hits == PROBE_ROWS and clean == PROBE_ROWS:
                chosen = cfg; break
        if chosen: MODEL_CONFIG[name] = chosen; print(f"  -> {name}: {chosen['tag']}\n")
        else: print(f"  -> {name}: no clean configuration; excluded and reported as format-limited\n")
    pd.DataFrame(probe_rows).to_csv(OUT7/"stage_a_probe.csv", index=False)
    json.dump({k: v["tag"] for k, v in MODEL_CONFIG.items()},
              open(OUT7/"stage_a_chosen_config.json", "w"), indent=2)

    # --- cost estimate before Stage B -----------------------------------------------------
    n_rows = len(three_way_u(U, SECOND_VARIANT_SEEDS[0])[2]) * len(SECOND_VARIANT_SEEDS)
    print(f"Stage B will score ~{n_rows:,} rows per model for {list(MODEL_CONFIG)}; "
          f"main campaign: 2,460 rows/model for USD 218.34 -> "
          f"expect ~USD {218.34*n_rows/2460*1.1*len(MODEL_CONFIG)/6:,.0f} for this panel")

    # --- Stage B: ten paired seeds, resumable ---------------------------------------------
    RESULTS_CSV, RAWJ = OUT7/"results.csv", OUT7/"raw_generations.jsonl"
    done = set()
    if RESULTS_CSV.exists():
        prev = pd.read_csv(RESULTS_CSV); done = set(zip(prev.seed, prev.model))
        print(f"resuming: {len(done)} pairs complete")
    def run_model(mid, cfg, icl, test, feats):
        preds, rawl, lat, errs, ti_s, to_s = [], [], [], 0, 0, 0
        for _, row in test.iterrows():
            s_, u_ = build_prompt(icl, row, feats); texts = []
            try:
                t0 = time.time(); txt, fr, ti, to = call(mid, s_, u_, cfg)
                lat.append(time.time()-t0)
                ti_s += ti; to_s += to; texts.append(txt); lab = parse_label(txt)
                if lab is None:
                    txt, fr, ti, to = call(mid, s_, u_ + "\n\nAnswer with the label only.", cfg)
                    ti_s += ti; to_s += to; texts.append(txt); lab = parse_label(txt)
            except Exception as e:
                errs += 1; lab = None
                texts.append(f"__error__ {type(e).__name__}: {str(e)[:110]}")
            preds.append(lab); rawl.append(texts)
        return preds, rawl, lat, errs, ti_s, to_s
    for seed in SECOND_VARIANT_SEEDS:
        dev, icl, test = three_way_u(U, seed); feats = mi_features(dev, K_FEATURES, seed)
        y_true = test["Label"].astype(str); batch = []
        print(f"\n=== seed {seed} | test={len(test)} | features={feats}")
        if (seed, "RandomForest") not in done:
            rf = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1)
            rf.fit(icl[feats].to_numpy(), icl["Label"].astype(str).to_numpy())
            p = rf.predict(test[feats].to_numpy())
            batch.append({"seed": seed, "model": "RandomForest", "kind": "anchor", "config": "n/a",
                          "accuracy": accuracy_score(y_true, p),
                          "macro_f1": f1_score(y_true, p, average="macro", zero_division=0),
                          "parse_rate": 1.0, "errors": 0, "usd": 0.0, "latency_median_s": 0.0,
                          "wall_min": 0.0, "n_eval": len(test)})
            with open(RAWJ, "a") as fh:
                fh.write(json.dumps({"seed": seed, "model": "RandomForest", "config": "n/a",
                                     "features": feats, "y_true": list(y_true),
                                     "y_pred": list(map(str, p)), "raw": []}) + "\n")
            print(f"  RandomForest       acc={batch[-1]['accuracy']:.4f}")
        for name, cfg in MODEL_CONFIG.items():
            if (seed, name) in done: print(f"  {name:<28} done, skipped"); continue
            t0 = time.time()
            preds, rawl, lat, errs, ti, to = run_model(RESOLVED[name], cfg, icl, test, feats)
            yp = pd.Series(preds).fillna("__unparsed__").astype(str)
            pin, pout = PRICES.get(name, (0.0, 0.0))
            batch.append({"seed": seed, "model": name, "kind": "llm", "config": cfg["tag"],
                          "accuracy": accuracy_score(y_true, yp),
                          "macro_f1": f1_score(y_true, yp, labels=LABELS, average="macro",
                                               zero_division=0),
                          "parse_rate": float(np.mean([p is not None for p in preds])),
                          "errors": errs, "usd": (ti*pin + to*pout)/1e6,
                          "latency_median_s": float(np.median(lat)) if lat else np.nan,
                          "wall_min": (time.time()-t0)/60, "n_eval": len(test)})
            with open(RAWJ, "a") as fh:
                fh.write(json.dumps({"seed": seed, "model": name, "config": cfg["tag"],
                                     "features": feats, "y_true": list(y_true),
                                     "y_pred": preds, "raw": rawl}) + "\n")
            print(f"  {name:<28} acc={batch[-1]['accuracy']:.4f} "
                  f"macroF1={batch[-1]['macro_f1']:.4f} "
                  f"parse={batch[-1]['parse_rate']:.3f} usd={batch[-1]['usd']:.2f}")
            pd.DataFrame(batch).to_csv(RESULTS_CSV, mode="a",
                                       header=not RESULTS_CSV.exists(), index=False)
            batch = []
        if batch:
            pd.DataFrame(batch).to_csv(RESULTS_CSV, mode="a",
                                       header=not RESULTS_CSV.exists(), index=False)
    print("\nExperiment 7 finished. Persist OUT_DIR before the session ends.")
else:
    print("Experiment 7 disabled.")

In [ ]:
# ============================================================
#  PATCH — Experiment 7: re-run ONLY the 404-contaminated pairs
#  (Qwen3.5-397B-A17B, seeds 2 and 3)
#
#  Place and run AFTER the switches / deps / repository / helper
#  cells (RUN_AXIS2_SECOND may stay False; this cell is
#  self-contained apart from load_base and OUT). It:
#    1. backs up results.csv and raw_generations.jsonl;
#    2. removes the two target rows ONLY if their raw generations
#       are transport errors (__error__), never model output;
#    3. re-verifies the frozen Qwen configuration on the five
#       blind probes (calibration seed 1000, as in Stage A);
#    4. rebuilds each seed's split and asserts it reproduces the
#       persisted anchor y_true EXACTLY before any paid call;
#    5. re-runs the two model-seed pairs and appends the records;
#    6. recomputes paired_tests.csv over the completed record.
# ============================================================
PATCH_PAIRS = [("Qwen3.5-397B-A17B", 2), ("Qwen3.5-397B-A17B", 3)]

from openai import OpenAI
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import itertools as _it

# Nebius migrated serving from AI Studio to Token Factory (old surface
# deprecated 2026-08-31); the model lives on the new surface. Each base is
# tried in order until the model actually serves there.
BASE_URLS = ["https://api.tokenfactory.nebius.com/v1/",
             "https://api.tokenfactory.us-central1.nebius.com/v1/",
             "https://api.studio.nebius.com/v1/"]
OUT7 = OUT/"axis2_second_variant"
RESULTS_CSV, RAWJ = OUT7/"results.csv", OUT7/"raw_generations.jsonl"
assert RESULTS_CSV.exists() and RAWJ.exists(), "run this next to the existing Experiment-7 record"

PATCH_MODEL = "Qwen3.5-397B-A17B"
TARGET_PATTERNS = ["qwen3.5-397b", "397b-a17b"]
FROZEN_CFG = {"tag": "nothink_template", "max_tokens": 4096, "prefill": False,
              "extra": {"chat_template_kwargs": {"thinking": False, "enable_thinking": False}}}
PRICE_IN, PRICE_OUT = 0.60, 3.60          # USD per 1M tokens, main campaign
K_FEATURES, K_SHOTS, N_TEST_PER_CLASS, N_DEV_PER_CLASS = 12, 5, 30, 400

# ---- protocol functions, verbatim from Experiment 7 --------------------------
U = load_base("unsw", False)
LABELS = sorted(U.Label.astype(str).unique())

def three_way_u(d, seed):
    rng = np.random.default_rng(seed); dev, icl, test = [], [], []
    for _, part in d.groupby("Label", sort=True):
        part = part.iloc[rng.permutation(len(part))]
        n_dev = min(N_DEV_PER_CLASS, max(0, len(part)-K_SHOTS-N_TEST_PER_CLASS))
        rest = len(part)-n_dev
        n_icl = min(K_SHOTS, rest); n_test = min(N_TEST_PER_CLASS, max(0, rest-n_icl))
        dev.append(part.iloc[:n_dev]); icl.append(part.iloc[n_dev:n_dev+n_icl])
        test.append(part.iloc[n_dev+n_icl:n_dev+n_icl+n_test])
    cat = lambda ps: pd.concat([p for p in ps if len(p)], ignore_index=True)
    return cat(dev), cat(icl), cat(test)

def mi_features(d, k, seed):
    x = d.drop(columns=["Label"], errors="ignore").select_dtypes(include=[np.number])
    y = LabelEncoder().fit_transform(d["Label"].astype(str))
    return list(x.columns[np.argsort(mutual_info_classif(x, y, random_state=seed))[::-1][:k]])

def build_prompt(icl, row, features):
    shots = "\n".join(json.dumps({**{f: round(float(r[f]), 4) for f in features},
                                  "label": str(r["Label"])})
                      for _, r in icl.iterrows())
    system = ("You are a network intrusion detection classifier. "
              f"Choose exactly one label from this list: {', '.join(LABELS)}. "
              "Respond with the label only, on its own line, with no explanation.")
    return system, f"{shots}\n" + json.dumps({f: round(float(row[f]), 4) for f in features})

def parse_label(text):
    if not text: return None
    t = re.sub(r"<(think|thinking|reasoning)>.*?</\1>", " ", text, flags=re.S|re.I)
    t = re.sub(r"^.*</(think|thinking|reasoning)>", " ", t, flags=re.S|re.I)
    for line in [l.strip(" \t\"'`*.-:") for l in t.strip().splitlines()[::-1] if l.strip()]:
        for lab in sorted(LABELS, key=len, reverse=True):
            if line.lower() == lab.lower(): return lab
        for lab in sorted(LABELS, key=len, reverse=True):
            if lab.lower() in line.lower(): return lab
    return None

def call(model_id, system, user, cfg, temperature=0.0):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    if cfg.get("prefill"): msgs.append({"role": "assistant", "content": "Label:"})
    kw = dict(model=model_id, messages=msgs, max_tokens=cfg["max_tokens"],
              temperature=temperature, timeout=300)
    if cfg.get("extra"): kw["extra_body"] = cfg["extra"]
    r = client.chat.completions.create(**kw); ch = r.choices[0]; u = getattr(r, "usage", None)
    return ((ch.message.content or ""), getattr(ch, "finish_reason", "?"),
            getattr(u, "prompt_tokens", 0) or 0, getattr(u, "completion_tokens", 0) or 0)

def run_model(mid, cfg, icl, test, feats):
    preds, rawl, lat, errs, ti_s, to_s = [], [], [], 0, 0, 0
    for _, row in test.iterrows():
        s_, u_ = build_prompt(icl, row, feats); texts = []
        try:
            t0 = time.time(); txt, fr, ti, to = call(mid, s_, u_, cfg)
            lat.append(time.time()-t0)
            ti_s += ti; to_s += to; texts.append(txt); lab = parse_label(txt)
            if lab is None:
                txt, fr, ti, to = call(mid, s_, u_ + "\n\nAnswer with the label only.", cfg)
                ti_s += ti; to_s += to; texts.append(txt); lab = parse_label(txt)
        except Exception as e:
            errs += 1; lab = None
            texts.append(f"__error__ {type(e).__name__}: {str(e)[:110]}")
        preds.append(lab); rawl.append(texts)
    return preds, rawl, lat, errs, ti_s, to_s

# ---- 1. resolve the endpoint (this is what 404'd during seeds 2-3) -----------
# The listing can contain dead aliases that 404 at inference (that is exactly
# what poisoned seeds 2-3), so every candidate is live-probed with a one-token
# call and the first id that actually serves is used.
MID_OVERRIDE = None            # set to an exact id to force it (checked on every base)
import openai as _openai
norm = lambda s: re.sub(r"[^a-z0-9.]", "", s.lower())
client, mid = None, None
for base in BASE_URLS:
    c = OpenAI(base_url=base, api_key=os.environ["NEBIUS_API_KEY"])
    try:
        available = sorted(m.id for m in c.models.list().data)
    except _openai.AuthenticationError:
        print(f"{base}: API key not accepted here (create a Token Factory key "
              f"at tokenfactory.nebius.com if this is the new surface)"); continue
    except Exception as e:
        print(f"{base}: listing failed ({type(e).__name__})"); continue
    candidates = ([MID_OVERRIDE] if MID_OVERRIDE else
                  [m for m in available if any(norm(p) in norm(m) for p in TARGET_PATTERNS)])
    print(f"{base}: candidates {candidates or '(none)'}")
    for cand in candidates:
        try:
            c.chat.completions.create(model=cand, max_tokens=1, timeout=60,
                                      messages=[{"role": "user", "content": "ping"}])
            client, mid = c, cand; print(f"  {cand}: serves"); break
        except _openai.NotFoundError:
            print(f"  {cand}: listed but 404s at inference, skipped")
        except Exception as e:   # rate limit, transient 5xx: the deployment exists
            client, mid = c, cand; print(f"  {cand}: alive ({type(e).__name__}), using it"); break
    if mid: break
assert mid, (f"no serving deployment found for {PATCH_MODEL} on any surface. "
             f"Check tokenfactory.nebius.com for the id and set MID_OVERRIDE, and "
             f"confirm the API key is valid for Token Factory")
print(f"endpoint resolved: {PATCH_MODEL} -> {mid}  (surface: {client.base_url})")

# ---- 2. quarantine the contaminated records (transport errors only) ----------
stamp = time.strftime("%Y%m%d-%H%M%S")
res = pd.read_csv(RESULTS_CSV)
raws = [json.loads(l) for l in open(RAWJ)]
targets = set(PATCH_PAIRS)
present = [(n, sd) for n, sd in PATCH_PAIRS
           if any(r["model"] == n and r["seed"] == sd for r in raws)
           or ((res.model == n) & (res.seed == sd)).any()]
if present:
    for name, seed in present:
        rec = next((r for r in raws if r["model"] == name and r["seed"] == seed), None)
        if rec is not None:
            flat = " ".join(str(t) for texts in rec["raw"] for t in texts)
            assert "__error__" in flat, (
                f"{name} seed {seed} contains genuine model output; refusing to overwrite")
    shutil.copy2(RESULTS_CSV, OUT7/f"results.csv.bak-{stamp}")
    shutil.copy2(RAWJ, OUT7/f"raw_generations.jsonl.bak-{stamp}")
    res = res[~res.apply(lambda r: (r["model"], r["seed"]) in targets, axis=1)]
    res.to_csv(RESULTS_CSV, index=False)
    with open(RAWJ, "w") as fh:
        for r in raws:
            if (r["model"], r["seed"]) not in targets:
                fh.write(json.dumps(r) + "\n")
    print(f"quarantined {len(present)} pairs (backups: *.bak-{stamp})")
else:
    print("target pairs already quarantined by a previous run; continuing")
# the anchor records are needed for the split-identity guard regardless
raws = [json.loads(l) for l in open(RAWJ)]

# ---- 3. Stage-A discipline: re-verify the frozen config on the blind probes --
devP, iclP, testP = three_way_u(U, 1000); featsP = mi_features(devP, K_FEATURES, 1000)
hits = clean = 0
for j in range(5):
    s_, u_ = build_prompt(iclP, testP.iloc[j], featsP)
    txt, fr, ti, to = call(mid, s_, u_, FROZEN_CFG)
    hits += int(parse_label(txt) is not None); clean += int(fr == "stop")
assert hits == 5 and clean == 5, f"frozen config no longer clean (parsed {hits}/5, clean {clean}/5)"
print(f"frozen configuration re-verified: {FROZEN_CFG['tag']} parsed 5/5, clean 5/5")

# ---- 4+5. re-run the two pairs with a split-identity guard -------------------
anchor_truth = {r["seed"]: r["y_true"] for r in raws if r["model"] == "RandomForest"}
for name, seed in PATCH_PAIRS:
    dev, icl, test = three_way_u(U, seed); feats = mi_features(dev, K_FEATURES, seed)
    y_true = test["Label"].astype(str)
    assert list(y_true) == anchor_truth[seed], (
        f"seed {seed}: rebuilt split does not reproduce the persisted anchor labels; aborting")
    print(f"\n=== seed {seed} | split verified against persisted anchor | test={len(test)}")
    t0 = time.time()
    preds, rawl, lat, errs, ti, to = run_model(mid, FROZEN_CFG, icl, test, feats)
    yp = pd.Series(preds).fillna("__unparsed__").astype(str)
    row = {"seed": seed, "model": name, "kind": "llm", "config": FROZEN_CFG["tag"],
           "accuracy": accuracy_score(y_true, yp),
           "macro_f1": f1_score(y_true, yp, labels=LABELS, average="macro", zero_division=0),
           "parse_rate": float(np.mean([p is not None for p in preds])),
           "errors": errs, "usd": (ti*PRICE_IN + to*PRICE_OUT)/1e6,
           "latency_median_s": float(np.median(lat)) if lat else np.nan,
           "wall_min": (time.time()-t0)/60, "n_eval": len(test)}
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    with open(RAWJ, "a") as fh:
        fh.write(json.dumps({"seed": seed, "model": name, "config": FROZEN_CFG["tag"],
                             "features": feats, "y_true": list(y_true),
                             "y_pred": preds, "raw": rawl}) + "\n")
    print(f"  {name} acc={row['accuracy']:.4f} macroF1={row['macro_f1']:.4f} "
          f"parse={row['parse_rate']:.3f} errors={errs} usd={row['usd']:.2f}")

# ---- 6. recompute the paired tests over the completed record -----------------
def _sf(d):
    d = np.asarray(d, float); o = abs(d.mean())
    return float(np.mean([abs(np.mean(d*np.array(x))) >= o-1e-15
                          for x in _it.product((-1, 1), repeat=len(d))]))
def _ci(d, n=50000):
    d = np.asarray(d, float); b = np.random.default_rng(0).choice(d, (n, len(d))).mean(1)
    return tuple(np.percentile(b, [2.5, 97.5]))
res = pd.read_csv(RESULTS_CSV)
assert res.groupby("model").seed.nunique().min() == 10, "record incomplete after patch"
assert res[res.model == PATCH_MODEL].parse_rate.min() > 0, "a zero-parse row remains"
rf = res[res.model == "RandomForest"].set_index("seed")
rows = []
for name, g in res[res.model != "RandomForest"].groupby("model"):
    g = g.set_index("seed").sort_index()
    d = (g.macro_f1 - rf.loc[g.index, "macro_f1"]).to_numpy()
    lo, hi = _ci(d)
    rows.append({"model": name, "accuracy": g.accuracy.mean(), "macro_f1": g.macro_f1.mean(),
                 "parse": g.parse_rate.mean(), "delta": d.mean(), "lo": lo, "hi": hi,
                 "wins": int((d > 0).sum()), "n": len(d), "p_raw": _sf(d),
                 "p_bonf": min(1.0, 6*_sf(d))})
PT = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
PT.to_csv(OUT7/"paired_tests.csv", index=False)
print("\n" + PT.round(4).to_string(index=False))
print("\nPatch complete. Re-run the Experiment-7 analysis cell (or the table builder)")
print("so table_axis2_second_variant.tex and the summary regenerate from this record,")
print("then re-archive OUT_DIR before the session ends.")

## Analysis — tables and prose numbers for the manuscript

Runs on whatever outputs exist in `OUT_DIR` (and the released record in `REPO_DIR`). Writes
LaTeX fragments to `OUT_DIR/generated/` and, if `MANUSCRIPT_DIR` is set, copies them into the
manuscript's `generated/` folder, replacing the red `\pending` stubs. The printed summary gives
every number referenced by a `\pending{...}` marker in the manuscript text.

In [ ]:
GEN = OUT/"generated"; GEN.mkdir(exist_ok=True)
SUMMARY = {}
def write(name, text):
    (GEN/name).write_text(text); print(f"[table] {name}")
    if MANUSCRIPT_DIR:
        d = Path(MANUSCRIPT_DIR)/"generated"; d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(GEN/name, d/name)

# ----------------------------------------------------------- Experiment 5: UNSW factorial
p5 = OUT/"exp5_size_by_balance_unsw.csv"
if p5.exists():
    E = pd.read_csv(p5)
    arms = E.groupby(["size","balance"]).agg(n_train=("n_train","mean"),
                                             n_unique=("n_unique_rows","mean"),
                                             macro_f1=("macro_f1","mean"),
                                             rare_recall=("rare_recall","mean"))
    pv = E.pivot(index="seed", columns="arm", values="rare_recall")
    contrasts = [("Balance, small budget","small_balanced","small_natural"),
                 ("Balance, large budget","large_balanced","large_natural"),
                 ("Size, natural distribution","large_natural","small_natural"),
                 ("Size, balanced distribution","large_balanced","small_balanced")]
    rows = []
    for lab, a, b in contrasts:
        d = (pv[a]-pv[b]).dropna().to_numpy(); lo, hi = ci(d, 50000)
        rows.append({"contrast": lab, "delta": d.mean(), "lo": lo, "hi": hi,
                     "wins": int((d>0).sum()), "n": len(d), "p": sf(d)})
    ph = holm([r["p"] for r in rows])
    for r, q in zip(rows, ph): r["p_holm"] = float(q)
    C = pd.DataFrame(rows); C.to_csv(OUT/"exp5_contrasts.csv", index=False)
    t = ["\\begin{minipage}{\\linewidth}","\\centering","\\textbf{Panel A: arm means.}\\\\[3pt]",
         "\\begin{tabular}{@{}llrrrr@{}}","\\toprule",
         "Budget & Distribution & Rows & Unique rows & Macro-$F_1$ & Rare-class recall \\\\","\\midrule"]
    for (sz, bl) in [("small","natural"),("small","balanced"),("large","natural"),("large","balanced")]:
        r = arms.loc[(sz, bl)]
        t.append(f"{sz.capitalize()} & {bl.capitalize()} & {int(round(r.n_train)):,} & "
                 f"{int(round(r.n_unique)):,} & {r.macro_f1:.3f} & {r.rare_recall:.3f} \\\\")
    t += ["\\bottomrule","\\end{tabular}","\\par\\medskip",
          "\\textbf{Panel B: planned paired contrasts.}\\\\[3pt]",
          "\\begin{tabular}{@{}lrrrr@{}}","\\toprule",
          "Contrast & $\\Delta$ rare-class recall & 95\\% CI & Wins & $p_{\\mathrm{Holm}}$ \\\\","\\midrule"]
    for _, r in C.iterrows():
        t.append(f"{r.contrast} & {r.delta:+.3f} & [{r.lo:+.3f}, {r.hi:+.3f}] & "
                 f"{r.wins}/{r.n} & {r.p_holm:.4f} \\\\")
    t += ["\\bottomrule","\\end{tabular}","\\end{minipage}"]
    write("table_size_balance_unsw.tex", "\n".join(t)+"\n")
    SUMMARY["exp5_unsw_factorial"] = {"arms": arms.reset_index().to_dict("records"),
                                      "contrasts": C.to_dict("records")}
else:
    print("[skip] experiment 5 outputs not found")

# ----------------------------------------------------------- Experiment 4b: ManyClass
p4 = OUT/"exp4_manyclass.csv"
if not p4.exists():
    alt = Path(REPO_DIR)/"results"/"sensitivity"/"exp4_manyclass.csv"
    p4 = alt if alt.exists() else None
if p4 is not None:
    MC = pd.read_csv(p4); rows = []
    for (a, r_), g in MC.groupby(["alphabet_size","n_estimators_redundancy"]):
        x = g.rare_recall.to_numpy()
        lo, hi = tci(x) if len(x) < 10 else ci(x, 50000)
        rows.append({"alphabet": a, "redundancy": r_, "n": len(x),
                     "accuracy": g.accuracy.mean(), "macro_f1": g.macro_f1.mean(),
                     "rare_recall": x.mean(), "lo": lo, "hi": hi})
    M4 = pd.DataFrame(rows)
    kind = "Student-$t$" if M4.n.min() < 10 else "percentile bootstrap"
    t = ["\\begin{tabular}{@{}rrrrrrr@{}}","\\toprule",
         "Alphabet & Redundancy & $n$ & Accuracy & Macro-$F_1$ & Rare-class recall & 95\\% CI \\\\",
         "\\midrule"]
    for _, r in M4.iterrows():
        t.append(f"{int(r.alphabet)} & {int(r.redundancy)} & {int(r.n)} & {r.accuracy:.4f} & "
                 f"{r.macro_f1:.4f} & {r.rare_recall:.4f} & [{r.lo:.4f}, {r.hi:.4f}] \\\\")
    t += ["\\bottomrule","\\end{tabular}"]
    write("table_manyclass.tex", "\n".join(t)+"\n")
    mp = MC.pivot(index="seed", columns=["alphabet_size","n_estimators_redundancy"],
                  values="rare_recall")
    paired = []
    for col in mp.columns:
        if col == (5, 1): continue
        d = (mp[col]-mp[(5,1)]).dropna().to_numpy()
        paired.append({"setting": f"a{col[0]}_r{col[1]}", "delta_vs_a5r1": d.mean(),
                       "p": sf(d), "n": len(d)})
    SUMMARY["manyclass"] = {"n_seeds": int(MC.seed.nunique()), "interval": kind,
                            "spread": float(M4.rare_recall.max()-M4.rare_recall.min()),
                            "settings": M4.to_dict("records"), "paired_vs_a5r1": paired}
    print(f"ManyClass: n_seeds={MC.seed.nunique()} "
          f"spread={M4.rare_recall.max()-M4.rare_recall.min():.4f} ({kind} intervals)")

# ----------------------------------------------------------- Experiment 6: data-rich references
p6 = OUT/"exp6_axis2_datarich.csv"
if p6.exists():
    E6 = pd.read_csv(p6)
    rf_ref = (pd.read_csv(Path(REPO_DIR)/"results"/"axis2"/"results_original_six_model_campaign.csv")
                .query("model=='RandomForest'").drop_duplicates("seed").set_index("seed"))
    repo_res = Path(REPO_DIR)/"results"/"derived"/"axis2_closed_label_per_seed.csv"
    best = None
    if repo_res.exists():
        per = pd.read_csv(repo_res)
        best_name = per.groupby("model").macro_f1_closed.mean().idxmax()
        best = per[per.model == best_name].set_index("seed")
    t = ["\\begin{tabular}{@{}lrrrrrr@{}}","\\toprule",
         "Reference & Fitting rows & Accuracy & Macro-$F_1$ & $\\Delta F_1$ vs.\\ anchor & Wins & $p$ \\\\",
         "\\midrule",
         f"Random Forest anchor (45 rows) & 45 & {rf_ref.accuracy.mean():.3f} & "
         f"{rf_ref.macro_f1.mean():.3f} & -- & -- & -- \\\\"]
    out = []
    for ref in ["development-rich","large natural"]:
        for learner in ["strong RF","XGBoost"]:
            g = E6[(E6.reference==ref)&(E6.learner==learner)].set_index("seed").sort_index()
            if not len(g): continue
            common = sorted(set(g.index)&set(rf_ref.index))
            d = g.loc[common,"macro_f1"].to_numpy()-rf_ref.loc[common,"macro_f1"].to_numpy()
            rec = {"reference": ref, "learner": learner, "n_fit": g.n_fit.mean(),
                   "accuracy": g.accuracy.mean(), "macro_f1": g.macro_f1.mean(),
                   "delta_vs_anchor": d.mean(), "wins": int((d>0).sum()), "n": len(d), "p": sf(d)}
            if best is not None:
                cb = sorted(set(g.index)&set(best.index))
                db = g.loc[cb,"macro_f1"].to_numpy()-best.loc[cb,"macro_f1_closed"].to_numpy()
                rec.update({"best_llm": best_name, "delta_vs_best_llm": db.mean(),
                            "wins_vs_best_llm": int((db>0).sum())})
            out.append(rec)
            lab = {"development-rich":"development partition",
                   "large natural":"large natural sample"}[ref]
            t.append(f"{learner.replace('strong RF','Strong RF')}, {lab} & "
                     f"{int(round(g.n_fit.mean())):,} & {g.accuracy.mean():.3f} & "
                     f"{g.macro_f1.mean():.3f} & {d.mean():+.3f} & {int((d>0).sum())}/{len(d)} & "
                     f"{sf(d):.4f} \\\\")
    t += ["\\bottomrule","\\end{tabular}"]
    write("table_axis2_datarich.tex", "\n".join(t)+"\n")
    SUMMARY["exp6_datarich"] = out
    print(pd.DataFrame(out).round(4).to_string())
else:
    print("[skip] experiment 6 outputs not found")

# ----------------------------------------------------------- Experiment 7: second variant
p7 = OUT/"axis2_second_variant"/"results.csv"
if p7.exists():
    from sklearn.metrics import accuracy_score, f1_score
    R7 = pd.read_csv(p7).drop_duplicates(["seed","model"], keep="last")
    raw7 = [json.loads(l) for l in
            (OUT/"axis2_second_variant"/"raw_generations.jsonl").read_text().splitlines()
            if l.strip()]
    rows = []
    for r in raw7:
        if r["model"] == "RandomForest": continue
        y = np.asarray(r["y_true"], dtype=str); labels = sorted(set(y))
        pred = np.asarray([p if p is not None else "__unparsed__" for p in r["y_pred"]], dtype=str)
        rows.append({"seed": int(r["seed"]), "model": r["model"],
                     "accuracy": accuracy_score(y, pred),
                     "macro_f1_closed": f1_score(y, pred, labels=labels, average="macro",
                                                 zero_division=0),
                     "parse_rate": float(np.mean([v is not None for v in r["y_pred"]]))})
    S7 = pd.DataFrame(rows).drop_duplicates(["seed","model"], keep="last")
    rf7 = R7[R7.model=="RandomForest"].set_index("seed")
    cost7 = (R7[R7.kind=="llm"].groupby("model")
               .agg(usd=("usd","sum"), rows=("n_eval","sum"), lat=("latency_median_s","median")))
    cost7["per1k"] = cost7.usd/cost7.rows*1000
    order = ["Kimi-K3","GLM-5.2","MiniMax-M3","Qwen3.5-397B-A17B","DeepSeek-V4-Pro",
             "Nemotron-3-Ultra-550b-a55b"]
    names = {"Kimi-K3":"Kimi-K3","GLM-5.2":"GLM-5.2","MiniMax-M3":"MiniMax-M3",
             "Qwen3.5-397B-A17B":"Qwen3.5","DeepSeek-V4-Pro":"DeepSeek-V4-Pro",
             "Nemotron-3-Ultra-550b-a55b":"Nemotron-3 Ultra"}
    tests = []
    for m in [m for m in order if m in set(S7.model)]:
        g = S7[S7.model==m].set_index("seed"); common = sorted(set(g.index)&set(rf7.index))
        d = g.loc[common,"macro_f1_closed"].to_numpy()-rf7.loc[common,"macro_f1"].to_numpy()
        lo, hi = ci(d, 50000)
        tests.append({"model": m, "accuracy": g.accuracy.mean(),
                      "macro_f1": g.macro_f1_closed.mean(), "parse": g.parse_rate.mean(),
                      "delta": d.mean(), "lo": lo, "hi": hi, "wins": int((d>0).sum()),
                      "n": len(d), "p_raw": sf(d)})
    nfam = len(tests)
    for r in tests: r["p_bonf"] = min(1.0, nfam*r["p_raw"])
    T7 = pd.DataFrame(tests); T7.to_csv(OUT/"axis2_second_variant"/"paired_tests.csv", index=False)
    t = ["\\begin{tabular}{@{}lrrrrrrr@{}}","\\toprule",
         "Model & Accuracy & Macro-$F_1$ & Parse & $\\Delta F_1$ & $p_{\\mathrm{Bonf}}$ & "
         "Cost (USD) & USD / 1{,}000 rows \\\\","\\midrule",
         f"Random Forest & {rf7.accuracy.mean():.3f} & {rf7.macro_f1.mean():.3f} & 1.000 & "
         f"-- & -- & 0.00 & 0.00 \\\\"]
    for _, r in T7.iterrows():
        c = cost7.loc[r.model] if r.model in cost7.index else None
        t.append(f"{names[r.model]} & {r.accuracy:.3f} & {r.macro_f1:.3f} & {r.parse:.3f} & "
                 f"{r.delta:+.3f} & {r.p_bonf:.4f} & "
                 f"{(c.usd if c is not None else float('nan')):.2f} & "
                 f"{(c.per1k if c is not None else float('nan')):.2f} \\\\")
    t += ["\\midrule", f"Total & & & & & & {cost7.usd.sum():.2f} & \\\\",
          "\\bottomrule","\\end{tabular}"]
    write("table_axis2_second_variant.tex", "\n".join(t)+"\n")
    rank_note = None
    main = Path(REPO_DIR)/"results"/"derived"/"axis2_closed_label_summary.csv"
    if main.exists():
        Mn = pd.read_csv(main).set_index("model").macro_f1_closed
        common = [m for m in T7.model if m in Mn.index]
        if len(common) >= 3:
            a = T7.set_index("model").loc[common,"macro_f1"]; b = Mn.loc[common]
            rank_note = float(pd.Series(a.values).corr(pd.Series(b.values), method="spearman"))
    SUMMARY["exp7_second_variant"] = {"tests": T7.to_dict("records"),
                                      "cost": cost7.reset_index().to_dict("records"),
                                      "anchor_accuracy": float(rf7.accuracy.mean()),
                                      "anchor_macro_f1": float(rf7.macro_f1.mean()),
                                      "spearman_vs_cic_ordering": rank_note, "n_family": nfam}
    print(T7.round(4).to_string()); print(cost7.round(3).to_string())
    print("Spearman vs CIC-IDS2017 ordering:", rank_note)
else:
    print("[skip] experiment 7 outputs not found")

(OUT/"revision4_summary.json").write_text(json.dumps(SUMMARY, indent=2, default=float))
print("\n=== numbers for the \\pending markers (revision4_summary.json) ===")
print(json.dumps(SUMMARY, indent=2, default=float)[:6000])

In [ ]:
# ============================================================
#  DOWNLOAD — archive everything this notebook produced.
#  Paste as the final cell. Safe to run at any point, including
#  part-way through a long experiment.
# ============================================================
import hashlib, json, os, zipfile
from datetime import datetime, timezone
from pathlib import Path

# Resolve the output directory even if this cell is run on its own.
try:
    OUT
except NameError:
    OUT = Path(os.environ.get("OUT_DIR", "revision4_outputs"))
OUT = Path(OUT)

INCLUDE_SOURCE = True    # bundle the notebook so the archive reproduces itself
INCLUDE_RAW    = True    # exp2_raw.jsonl holds every generation; large but required for independent re-scoring
MAX_MB         = 512

if not OUT.exists() or not any(OUT.rglob("*")):
    print(f"[archive] nothing to package: {OUT.resolve()} is empty.")
    print("          Enable an experiment and run it first.")
else:
    files = sorted(p for p in OUT.rglob("*")
                   if p.is_file() and p.suffix not in (".zip",) and
                      (INCLUDE_RAW or p.suffix != ".jsonl"))
    # The cached Axis-2 table is re-derivable from Kaggle; exclude it from the archive.
    files = [p for p in files if p.name != "axis2_table.parquet"]

    extras = []
    if INCLUDE_SOURCE:
        for name in ("revision4_experiments.ipynb", "scripts/recompute_manuscript_results.py"):
            for base in (Path.cwd(), Path("/content")):
                c = (base / name)
                if c.is_file() and c.resolve() not in extras:
                    extras.append(c.resolve()); break

    total_mb = sum(p.stat().st_size for p in files + extras) / 1e6
    print(f"[archive] {len(files)} result files + {len(extras)} source files, {total_mb:.2f} MB")

    if total_mb > MAX_MB:
        print(f"[archive] ABORTED: exceeds MAX_MB={MAX_MB}. Largest files:")
        for p in sorted(files, key=lambda q: -q.stat().st_size)[:8]:
            print(f"            {p.stat().st_size/1e6:8.1f} MB  {p.relative_to(OUT)}")
        print("          Raise MAX_MB, or set INCLUDE_RAW = False to drop the generations.")
    else:
        # SHA-256 manifest, matching the discipline of the experiment record, so a
        # recipient can verify that what they received is what was produced.
        manifest = {"generated_utc": datetime.now(timezone.utc).isoformat(),
                    "output_dir": str(OUT.resolve()),
                    "experiments": {}, "n_files": len(files), "files": []}
        for flag, name in (("RUN_2X2_UNSW","exp5_size_by_balance_unsw.csv"),
                           ("RUN_AXIS2_DATARICH","exp6_axis2_datarich.csv"),
                           ("RUN_MANYCLASS_10","exp4_manyclass.csv"),
                           ("RUN_AXIS2_SECOND","axis2_second_variant/results.csv")):
            manifest["experiments"][flag] = {
                "enabled": bool(globals().get(flag, False)),
                "output_present": (OUT/name).is_file()}
        for p in files + extras:
            h = hashlib.sha256()
            with p.open("rb") as fh:
                for b in iter(lambda: fh.read(1 << 20), b""):
                    h.update(b)
            manifest["files"].append({
                "path": str(p.relative_to(OUT)) if p.is_relative_to(OUT) else f"source/{p.name}",
                "bytes": p.stat().st_size, "sha256": h.hexdigest()})
        mpath = OUT / "ARCHIVE_MANIFEST.json"
        mpath.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

        stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
        archive = Path(f"sensitivity_experiments_{stamp}.zip")
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
            for p in files + [mpath]:
                z.write(p, arcname=str(Path("revision4_outputs") / p.relative_to(OUT)))
            for p in extras:
                z.write(p, arcname=f"source/{p.name}")

        by_ext = {}
        for p in files:
            by_ext[p.suffix or "(none)"] = by_ext.get(p.suffix or "(none)", 0) + 1
        print(f"[archive] wrote {archive}  ({archive.stat().st_size/1e6:.2f} MB, "
              f"{len(files)+len(extras)+1} entries)")
        print("[archive] contents: " + ", ".join(f"{k} x{v}" for k, v in
                                                 sorted(by_ext.items(), key=lambda kv: -kv[1])))
        for flag, info in manifest["experiments"].items():
            if info["enabled"] and not info["output_present"]:
                print(f"[archive] WARNING: {flag} was enabled but produced no output file")

        delivered = False
        try:
            from google.colab import files as _cf
            _cf.download(str(archive)); delivered = True
            print("[archive] browser download started.")
        except Exception:
            pass
        if not delivered:
            try:
                from IPython.display import FileLink, display
                display(FileLink(str(archive)))
            except Exception:
                pass
        print(f"[archive] absolute path: {archive.resolve()}")